# Stable-RouteNet v4b — Enhanced Architecture (15-epoch run, 50% data)

Route by representation, assess trust separately, train the exact evaluation graph.

Evidence-stability-conditioned sparse expert routing with shared + specialist experts,
prototype-based representation-first router, Switch-style load balancing, and router
z-loss for generalizable deepfake detection.

**Data:** 420 training videos (FF++ 120 + Celeb-DF Celeb-real 120 + Celeb-DF YouTube-real 120),
90 validation videos, test on DFDCP + Celeb-DF-v2 + FF++ 6 manipulation families.
Fakes are generated on-the-fly by DD-SBI with diverse masks and Poisson blending;
**no real manipulated video is ever trained on.**

**v4b improvements over v4:**
- Phase 1: 3-domain training (FF++, Celeb-DF Celeb-real, YouTube-real), 900 videos
- Phase 2: Diverse SBI masks (full/upper/lower/middle/ellipse), Poisson blending, training degradations
- Phase 3: LoRA adaptation in DINOv2, frequency-domain (DCT) expert, forgery-type auxiliary branch
- Phase 4: Progressive SBI curriculum, contractive-repulsive loss
- Phase 5: Multiple metrics (AUROC, EER, AP, F1, bootstrap CI), 9 test conditions

---

## v4 design principles

1. **Route by forensic representation** — prototype router selects experts by cosine
   similarity to learned forensic attribute prototypes, not by a generic MLP that can
   discover domain shortcuts.
2. **Assess trust separately** — M/S/R measure evidence trust; routing identity is
   decoupled from trust to prevent cascading error.
3. **Preserve common knowledge** — a shared expert is always evaluated so common
   forensic structure remains available even when sparse routing is imperfect.
4. **Train the exact evaluation graph** — all schedules are relative to total steps;
   no module required for final evaluation is permanently disabled by an absolute
   threshold.

## v4 ablation plan

| Ablation | Change | Question |
|----------|--------|----------|
| A0 | Corrected v3.1 curriculum only | Is train/test graph mismatch the main failure? |
| A1 | A0 + Switch balance + z-loss | Does router stabilization prevent collapse? |
| A2 | A1 + shared expert | Does common knowledge stabilize transfer? |
| A3 | A2 + prototype router | Does representation-first routing reduce shortcuts? |
| A4 | A3 + semantic specialist axes | Does meaningful specialization improve OOD? |

This notebook implements the full v4 architecture. Set `ABLATION` in the config cell
to run specific ablation stages.

---


## Protocol and leakage control (spec 34, 35)

| Split | Source | Used for |
|---|---|---|
| **Train** | FF++ YouTube-c23 real x120, Celeb-DF-v2 Celeb-real x120, YouTube-real x120, DD-SBI fakes on-the-fly | gradient updates |
| **Val** (`val-hard`) | FF++ real x30, Celeb-DF real x60 (30 celeb-real + 30 YouTube-real), held out at video level, hard DD-SBI parameters | loss monitoring, threshold calibration, checkpoint selection |
| **Test — cross-dataset** | DFDCP 200 real + 200 fake | reported once, after freeze |
| **Test — cross-dataset** | Celeb-DF-v2 200 real + 200 fake | reported once, after freeze |
| **Test — cross-manipulation** | FF++ Deepfakes / Face2Face / FaceShifter / FaceSwap / NeuralTextures / DeepFakeDetection (200 each) | reported once, after freeze |

Enforced in code by `assert_no_leakage()`: train, val and test video identities are disjoint, and
no test video is ever passed to the DD-SBI generator. The final evaluation cell is the only place
test data is read, and it runs after `checkpoint_best.pt` is written.

Run order: config -> preflight (must print `PREFLIGHT PASSED`) -> training -> report/gates ->
final evaluation.


In [1]:
# ============================ CONFIG — STABLE-ROUTENET V4 ============================
import os

# --- run identity -----------------------------------------------------------
RUN_NAME     = "v4_b_5ep"
ABLATION     = "A4"        # A0 correct curriculum | A1 +switch/z | A2 +shared | A3 +prototype | A4 +semantic
SEED         = 0
DATA_ROOT    = "/kaggle/input/datasets/sekhar826/srn-v4b-data"
OUT_DIR      = f"/kaggle/working/outputs/{RUN_NAME}"
RESUME_FROM_LAST = False

# --- token geometry ---------------------------------------------------------
IMG, PATCH   = 392, 14
GRID         = IMG // PATCH          # 28 -> 784 patch tokens
T_FRAMES     = 8                     # frames per training clip
T_TEST       = 32                    # frames per test video

# --- data splits (3 domains, leave room for test) ---------------------------
# On disk: FF++ real 200 | Celeb-DF Celeb-real 200 | Celeb-DF YouTube-real 200
# Per source: 60 train + 30 val + 50 test = 140 (50% train reduction for more epochs)
N_TRAIN      = {"ffpp": 60,  "cdf": 120}    # 60 FF++ + 60 celeb-real + 60 YouTube-real
N_VAL        = {"ffpp": 30,  "cdf": 60}     # 30 FF++ + 30 celeb-real + 30 YouTube-real
TEST_N       = 50                            # videos per test condition (real or fake)

# --- optimisation -----------------------------------------------------------
EPOCHS       = 5
BATCH_VIDEOS = 2
ACCUM_STEPS  = 2                     # effective batch = 4 videos
LR_PEAK      = 1.5e-4
LR_FLOOR     = 0.05
WD           = 0.01
WARMUP_FRAC  = 0.10                  # warmup as fraction of total steps
GRAD_CLIP    = 1.0
EMA_DECAY    = 0.999
LABEL_SMOOTH = 0.05
MAX_STEPS    = None

# --- model ------------------------------------------------------------------
D_MODEL      = 384
FREQ_EXPERT     = True
N_EXPERTS, TOPK = (5 if FREQ_EXPERT else 4), 2   # 4 spatial + 1 frequency expert
EXPERT_HIDDEN = 512
EPS          = 1e-6
TAU_S        = 0.75
PROTOTYPE_DIM = D_MODEL              # prototype embedding dimension

# --- v4: relative curriculum (fractions of total planned steps) -------------
# All schedules defined as fractions of total training steps.
# This fixes the v3.1 critical bug where MoE losses activate after training ends.
PHASE_FRACTIONS = dict(
    foundation_end  = 0.20,   # Phase 1: det + loc + mass + shared warm-start
    evidence_end    = 0.40,   # Phase 2: + M/S targets, stability, interventions
    moe_end         = 0.60,   # Phase 3: + top-2 routing, shared, Switch balance, z-loss
    reliability_end = 0.80,   # Phase 4: + A/C/R, evidence weighting, alpha learning
    # Phase 5 (80-100%): consolidation, all objectives, reduced auxiliary
)
RAMP_FRAC    = 0.05               # linear ramp at each phase boundary (as fraction)

# --- v4b: SBI diversity ---------------------------------------------------
POISSON_BLEND   = True            # use Poisson (seamless) blending when available
TRAIN_DEGRADE   = True            # apply degradations to BOTH views during training

# --- v4b: LoRA in DINOv2 backbone ----------------------------------------
LORA_RANK       = 8               # LoRA rank for DINOv2 adaptation
LORA_ALPHA      = 16              # LoRA alpha scaling

# --- v4b: Frequency-domain expert ----------------------------------------

# --- v4b: Forgery-type auxiliary branch -----------------------------------
FORGERY_BRANCH  = True            # auxiliary classification head (6 FF++ types + real)
N_FORGERY_TYPES = 7               # 6 manipulation families + real

# --- v4b: Progressive SBI curriculum -------------------------------------
PROG_CURRICULUM = True            # easy -> medium -> hard SBI over epochs

# --- v4b: Contractive-repulsive loss -------------------------------------
CRO_LOSS        = True            # pull same-class together, push different apart
LAM_CRO         = 0.05            # CRO loss weight

# --- objective weights ------------------------------------------------------
LAM = dict(
    loc  = 1.00,
    mass = 0.25,
    stab = 0.20,
    s    = 0.20,
    s_dist = 0.05,
    sep  = 0.02,
    route = 0.05,
    bal  = 0.05,       # Switch-style load balance
    z    = 0.01,       # Router z-loss
    div  = 0.02,       # Expert diversity (prototype-based)
    nuis = 0.10,       # F_var nuisance prediction
    cro  = 0.05,       # Contractive-repulsive loss
)

# --- evaluation -------------------------------------------------------------
ROBUST_CONDS = ["clean", "jpeg30", "blur15", "resize50"]
SELECT_METRIC = "worst_auc"
BOOTSTRAP_N  = 2000
NUM_WORKERS  = 4

# --- ablation -> switches ---------------------------------------------------
_ABL = {
    # A0: corrected v3.1 curriculum only (no switch/z, no shared, no prototype)
    "A0": dict(switch=False, z_loss=False, shared=False, prototype=False,
               semantic=False, off=("z",)),
    # A1: + Switch-style balance + z-loss
    "A1": dict(switch=True, z_loss=True, shared=False, prototype=False,
               semantic=False, off=()),
    # A2: + shared expert
    "A2": dict(switch=True, z_loss=True, shared=True, prototype=False,
               semantic=False, off=()),
    # A3: + prototype router
    "A3": dict(switch=True, z_loss=True, shared=True, prototype=True,
               semantic=False, off=()),
    # A4: + semantic specialist axes
    "A4": dict(switch=True, z_loss=True, shared=True, prototype=True,
               semantic=True, off=()),
}
assert ABLATION in _ABL, f"ABLATION must be one of {sorted(_ABL)}"
ABL = _ABL[ABLATION]
for _k in ABL["off"]:
    LAM[_k] = 0.0

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/evaluation", exist_ok=True)

CFG = dict(RUN_NAME=RUN_NAME, ABLATION=ABLATION, ABL=ABL, SEED=SEED, DATA_ROOT=DATA_ROOT,
           OUT_DIR=OUT_DIR, IMG=IMG, PATCH=PATCH, GRID=GRID, T_FRAMES=T_FRAMES, T_TEST=T_TEST,
           N_TRAIN=N_TRAIN, N_VAL=N_VAL, TEST_N=TEST_N, EPOCHS=EPOCHS, BATCH_VIDEOS=BATCH_VIDEOS,
           ACCUM_STEPS=ACCUM_STEPS, LR_PEAK=LR_PEAK, LR_FLOOR=LR_FLOOR, WD=WD,
           WARMUP_FRAC=WARMUP_FRAC, GRAD_CLIP=GRAD_CLIP, EMA_DECAY=EMA_DECAY,
           LABEL_SMOOTH=LABEL_SMOOTH, D_MODEL=D_MODEL, N_EXPERTS=N_EXPERTS, TOPK=TOPK,
           EXPERT_HIDDEN=EXPERT_HIDDEN, EPS=EPS, TAU_S=TAU_S,
           PROTOTYPE_DIM=PROTOTYPE_DIM, PHASE_FRACTIONS=PHASE_FRACTIONS, RAMP_FRAC=RAMP_FRAC,
           LAM=LAM, ROBUST_CONDS=ROBUST_CONDS, SELECT_METRIC=SELECT_METRIC,
           POISSON_BLEND=POISSON_BLEND, TRAIN_DEGRADE=TRAIN_DEGRADE,
           LORA_RANK=LORA_RANK, LORA_ALPHA=LORA_ALPHA,
           FREQ_EXPERT=FREQ_EXPERT, FORGERY_BRANCH=FORGERY_BRANCH, N_FORGERY_TYPES=N_FORGERY_TYPES,
           PROG_CURRICULUM=PROG_CURRICULUM, CRO_LOSS=CRO_LOSS, LAM_CRO=LAM_CRO)

print(f"run={RUN_NAME} ablation={ABLATION} -> {ABL}")
print(f"epochs={EPOCHS} | active loss terms: {sorted(k for k, v in LAM.items() if v > 0)}")
print(f"phase fractions: {PHASE_FRACTIONS}")


run=v4_b_5ep ablation=A4 -> {'switch': True, 'z_loss': True, 'shared': True, 'prototype': True, 'semantic': True, 'off': ()}
epochs=5 | active loss terms: ['bal', 'cro', 'div', 'loc', 'mass', 'nuis', 'route', 's', 's_dist', 'sep', 'stab', 'z']
phase fractions: {'foundation_end': 0.2, 'evidence_end': 0.4, 'moe_end': 0.6, 'reliability_end': 0.8}


In [2]:
# ============================ IMPORTS + DETERMINISM ============================
import io, json, math, glob, time, random, shutil, hashlib, collections
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFilter, ImageEnhance, ImageDraw
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as TVT
from transformers import AutoModel
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score, balanced_accuracy_score,
                             confusion_matrix, roc_curve, precision_recall_curve)

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
N_GPU  = torch.cuda.device_count()
USE_DP = N_GPU > 1
AMP    = torch.float16
assert N_GPU in (0, 1, 2), f"expected up to 2 GPUs, found {N_GPU}"

def worker_init(wid):
    s = SEED * 1000 + wid
    random.seed(s); np.random.seed(s)

IMNET = TVT.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMNET_STD  = np.array([0.229, 0.224, 0.225], np.float32)

print("device:", device, "| GPUs:", N_GPU, "| DataParallel:", USE_DP, "| torch:", torch.__version__)
for i in range(N_GPU):
    print(f"  gpu{i}: {torch.cuda.get_device_name(i)}")


device: cuda | GPUs: 2 | DataParallel: True | torch: 2.10.0+cu128
  gpu0: Tesla T4
  gpu1: Tesla T4


In [3]:
# ============================ DATA DISCOVERY, SPLITS, LEAKAGE CONTROL ============================
def find_dir(root, name):
    hits = [p for p in Path(root).rglob(name) if p.is_dir()]
    return hits[0] if hits else None

def _has_layout(path):
    if not path.is_dir():
        return False
    names = {p.name.casefold() for p in path.iterdir() if p.is_dir()}
    return "celeb-df-v2" in names and ("dfdcp" in names or "faceforensics++" in names)

def resolve_dataset_root():
    """Locate the Stable-RouteNet-1 pack wherever Kaggle mounted it."""
    input_root = Path("/kaggle/input")
    candidates = [Path(DATA_ROOT)]
    if input_root.exists():
        candidates.append(input_root)
        candidates += sorted(p for p in input_root.iterdir() if p.is_dir())
        candidates += sorted(p.parent for p in input_root.rglob("Celeb-DF-v2") if p.is_dir())
    seen = set()
    for c in candidates:
        c = c.resolve()
        if str(c) in seen:
            continue
        seen.add(str(c))
        if _has_layout(c):
            return str(c)
    avail = sorted(str(p) for p in input_root.iterdir()) if input_root.exists() else []
    raise AssertionError(f"could not resolve the Stable-RouteNet-1 layout; available inputs: {avail}")

DATA_ROOT = resolve_dataset_root()
FFPP_REAL = find_dir(DATA_ROOT, "youtube")
CDF_ROOT  = find_dir(DATA_ROOT, "Celeb-DF-v2")
DFDCP_ROOT = find_dir(DATA_ROOT, "DFDCP")
FFPP_ROOT  = find_dir(DATA_ROOT, "FaceForensics++")
assert FFPP_REAL is not None and CDF_ROOT is not None, "FF++ real / Celeb-DF-v2 not found"
print(f"dataset root: {DATA_ROOT}")

def video_frame_dirs(real_root):
    """-> [(video_id, frames_dir, landmarks_dir_or_None)] sorted deterministically."""
    out = []
    if real_root is None or not Path(real_root).exists():
        return out
    for fr in sorted(Path(real_root).rglob("frames")):
        lm = fr.parent / "landmarks"
        for vd in sorted(fr.iterdir()):
            if vd.is_dir():
                out.append((vd.name, str(vd), str(lm / vd.name) if lm.exists() else None))
    return out

ffpp_all = video_frame_dirs(FFPP_REAL)
cdf_celeb_all = video_frame_dirs(CDF_ROOT / "Celeb-real")
cdf_yt_all    = video_frame_dirs(CDF_ROOT / "YouTube-real")
cdf_all = cdf_celeb_all + cdf_yt_all
print(f"on disk: FF++ real {len(ffpp_all)} | Celeb-DF Celeb-real {len(cdf_celeb_all)} | "
      f"Celeb-DF YouTube-real {len(cdf_yt_all)}")
assert len(ffpp_all) >= N_TRAIN["ffpp"] + N_VAL["ffpp"], "not enough FF++ real videos"
assert len(cdf_all)  >= N_TRAIN["cdf"]  + N_VAL["cdf"],  "not enough Celeb-DF real videos"

# deterministic video-level split: first N sorted, frozen in the manifest
# Celeb-DF: interleave Celeb-real and YouTube-real for domain diversity
cdf_half = N_TRAIN["cdf"] // 2
cdf_train_pool = cdf_celeb_all[:cdf_half] + cdf_yt_all[:N_TRAIN["cdf"] - cdf_half]
cdf_val_pool   = cdf_celeb_all[cdf_half:cdf_half + N_VAL["cdf"] // 2] + \
                 cdf_yt_all[N_TRAIN["cdf"] - cdf_half: N_TRAIN["cdf"] - cdf_half + N_VAL["cdf"] - N_VAL["cdf"] // 2]
train_vids = {"ffpp": ffpp_all[:N_TRAIN["ffpp"]],
              "cdf":  cdf_train_pool}
val_vids   = {"ffpp": ffpp_all[N_TRAIN["ffpp"]:N_TRAIN["ffpp"] + N_VAL["ffpp"]],
              "cdf":  cdf_val_pool}

# test sets: drawn AFTER train+val from the same sorted pools (disjoint by construction)
TEST_SETS = {}
if DFDCP_ROOT is not None:
    TEST_SETS["dfdcp_real"] = video_frame_dirs(DFDCP_ROOT / "original_videos")[:TEST_N]
    TEST_SETS["dfdcp_fake"] = (video_frame_dirs(DFDCP_ROOT / "method_A")
                               + video_frame_dirs(DFDCP_ROOT / "method_B"))[:TEST_N]
if CDF_ROOT is not None:
    # celeb-real: train=[0:120], val=[120:150], test=[150:200]
    cdf_celeb_offset = N_TRAIN["cdf"] // 2 + N_VAL["cdf"] // 2
    TEST_SETS["cdf_test_real"] = video_frame_dirs(CDF_ROOT / "Celeb-real")[cdf_celeb_offset:][:TEST_N]
    TEST_SETS["cdf_test_fake"] = video_frame_dirs(CDF_ROOT / "Celeb-synthesis")[:TEST_N]
if FFPP_ROOT is not None:
    # ffpp: train=[0:120], val=[120:150], test=[150:200]
    TEST_SETS["ffpp_test_real"] = ffpp_all[N_TRAIN["ffpp"] + N_VAL["ffpp"]:][:TEST_N]
    for manip in ["Deepfakes", "Face2Face", "FaceShifter", "FaceSwap",
                  "NeuralTextures", "DeepFakeDetection"]:
        got = video_frame_dirs(FFPP_ROOT / "manipulated_sequences" / manip / "c23")
        if got:
            TEST_SETS[f"ffpp_{manip}"] = got[:TEST_N]

def _keys(groups):
    return {f"{d}/{v[0]}" for d, vs in groups.items() for v in vs}

def assert_no_leakage():
    """spec 35: train / val / test identities disjoint; SBI never sees test videos."""
    tr, va = _keys(train_vids), _keys(val_vids)
    assert not (tr & va), f"train/val overlap: {sorted(tr & va)[:5]}"
    tr_ids = {k.split('/', 1)[1] for k in tr}
    va_ids = {k.split('/', 1)[1] for k in va}
    for name, recs in TEST_SETS.items():
        te_ids = {r[0] for r in recs}
        assert not (te_ids & tr_ids), f"{name} leaks into train: {sorted(te_ids & tr_ids)[:5]}"
        assert not (te_ids & va_ids), f"{name} leaks into val: {sorted(te_ids & va_ids)[:5]}"
    return True

assert_no_leakage()
MANIFEST = dict(
    data_root=DATA_ROOT,
    train={d: [v[0] for v in vs] for d, vs in train_vids.items()},
    val={d: [v[0] for v in vs] for d, vs in val_vids.items()},
    test={k: [r[0] for r in v] for k, v in TEST_SETS.items()},
)
MANIFEST["hash"] = hashlib.sha1(json.dumps(MANIFEST, sort_keys=True).encode()).hexdigest()[:12]
CFG["DATA_ROOT_RESOLVED"] = DATA_ROOT
CFG["MANIFEST_HASH"] = MANIFEST["hash"]
Path(OUT_DIR, "manifest.json").write_text(json.dumps(MANIFEST, indent=2))
Path(OUT_DIR, "config.json").write_text(json.dumps(CFG, indent=2, default=str))

print(f"train {sum(len(v) for v in train_vids.values())} | val {sum(len(v) for v in val_vids.values())} videos")
print("test subsets: " + ", ".join(f"{k}={len(v)}" for k, v in TEST_SETS.items()))
print(f"leakage check passed | manifest {MANIFEST['hash']}")


dataset root: /kaggle/input/datasets/sekhar826/srn-v4b-data
on disk: FF++ real 200 | Celeb-DF Celeb-real 200 | Celeb-DF YouTube-real 200
train 180 | val 90 videos
test subsets: dfdcp_real=50, dfdcp_fake=50, cdf_test_real=50, cdf_test_fake=50, ffpp_test_real=50, ffpp_Deepfakes=50, ffpp_Face2Face=50, ffpp_FaceShifter=50, ffpp_FaceSwap=50, ffpp_NeuralTextures=50, ffpp_DeepFakeDetection=50
leakage check passed | manifest 01c542bb0852


In [4]:
# ============================ DD-SBI GENERATOR (v3) + INTERVENTIONS ============================
# Wider forgery distribution than v1/v2 to attack the compositing shortcut that produced
# val AUROC 1.0 / DFDCP 0.57: sub-region masks, jittered mask geometry, cross-frame blend
# sources, continuous blend ratio, source-side JPEG. Reals are quality-aligned with the
# same global jitter the blend source receives.

NUISANCE = ["jpeg", "blur", "resize", "photo"]
NUIS_IDX = {k: i for i, k in enumerate(NUISANCE)}
MASK_REGIONS = ["full", "upper", "lower", "middle"]

def photometric_jitter(img, rng):
    """Global appearance jitter. Applied to SBI sources AND to reals (quality alignment)."""
    if rng.random() < 0.8:
        img = ImageEnhance.Color(img).enhance(rng.uniform(0.6, 1.4))
    if rng.random() < 0.8:
        img = ImageEnhance.Brightness(img).enhance(rng.uniform(0.7, 1.3))
    if rng.random() < 0.6:
        img = ImageEnhance.Contrast(img).enhance(rng.uniform(0.8, 1.2))
    if rng.random() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0.2, 1.0)))
    if rng.random() < 0.3:
        buf = io.BytesIO(); img.save(buf, "JPEG", quality=rng.randint(60, 95)); buf.seek(0)
        img = Image.open(buf).convert("RGB")
    return img

def _band(size, region, rng):
    """Soft horizontal band used to carve sub-regions out of the face polygon."""
    w, h = size
    if region == "full":
        return None
    m = Image.new("L", size, 0)
    if region == "upper":
        y0, y1 = 0.0, rng.uniform(0.45, 0.62)
    elif region == "lower":
        y0, y1 = rng.uniform(0.38, 0.55), 1.0
    else:
        y0, y1 = rng.uniform(0.25, 0.35), rng.uniform(0.65, 0.78)
    ImageDraw.Draw(m).rectangle([0, int(h * y0), w, int(h * y1)], fill=255)
    return m.filter(ImageFilter.GaussianBlur(h * 0.03))

def landmark_mask(lm_path, size, rng, region="full", jitter=0.0):
    """Face polygon from landmarks, optionally jittered and restricted to a sub-region."""
    m = Image.new("L", size, 0)
    ok = False
    if lm_path is not None:
        try:
            lm = np.load(lm_path).astype(np.float32).reshape(-1, 2)
            pts = lm * np.array([size[0] / 256.0, size[1] / 256.0], np.float32)
            if jitter > 0:
                pts = pts + np.random.normal(0.0, jitter * size[0], pts.shape).astype(np.float32)
            if len(pts) >= 3:
                ImageDraw.Draw(m).polygon([tuple(p) for p in pts], fill=255)
                ok = True
        except Exception:
            ok = False
    if not ok:
        w, h = size
        cx, cy = w * 0.5 + rng.uniform(-0.03, 0.03) * w, h * 0.5 + rng.uniform(-0.03, 0.03) * h
        rx, ry = w * rng.uniform(0.34, 0.42), h * rng.uniform(0.40, 0.47)
        ImageDraw.Draw(m).ellipse([cx - rx, cy - ry, cx + rx, cy + ry], fill=255)
    band = _band(size, region, rng)
    if band is not None:
        m = Image.fromarray((np.array(m, np.float32) * np.array(band, np.float32) / 255.0)
                            .clip(0, 255).astype(np.uint8))
    return m

def make_sbi(img_pil, lm_path=None, src_pil=None, hard=False, rng=None):
    """Self-blended image + exact GT mask (spec 18). Returns (uint8 HWC, float mask HW)."""
    rng = rng or random
    W, H = img_pil.size
    src = (src_pil if src_pil is not None else img_pil).copy()
    if src.size != (W, H):
        src = src.resize((W, H), Image.BILINEAR)
    src = photometric_jitter(src, rng)
    # small affine offset so the blend is not pixel-aligned
    s = rng.uniform(0.90, 1.08)
    sw, sh = max(8, int(W * s)), max(8, int(H * s))
    canvas = Image.new("RGB", (W, H))
    canvas.paste(src.resize((sw, sh), Image.BILINEAR),
                 (rng.randint(min(0, W - sw), max(0, W - sw)),
                  rng.randint(min(0, H - sh), max(0, H - sh))))
    src = canvas

    region = rng.choice(MASK_REGIONS[1:] if hard else MASK_REGIONS)
    mask = landmark_mask(lm_path, (W, H), rng, region=region, jitter=rng.uniform(0.0, 0.012))
    mask = mask.filter(ImageFilter.GaussianBlur(rng.uniform(2.0, 8.0)))
    arr = np.array(mask, np.float32) / 255.0
    ratio = rng.uniform(0.15, 0.45) if hard else rng.uniform(0.15, 1.0)
    arr = arr * ratio

    if (arr > 0.02).mean() < 0.004:                      # degenerate polygon -> ellipse fallback
        m2 = landmark_mask(None, (W, H), rng, region=region)
        arr = np.array(m2.filter(ImageFilter.GaussianBlur(3)), np.float32) / 255.0 * ratio

    a = arr[..., None]
    t_img = np.asarray(img_pil, np.float32)
    t_src = np.asarray(src, np.float32)
    blended = np.clip(t_src * a + t_img * (1.0 - a), 0, 255).astype(np.uint8)

    # Phase 2C: Poisson (seamless) blending for diverse artifacts
    if POISSON_BLEND and rng.random() < 0.3 and (arr > 0.02).mean() > 0.01:
        try:
            center = (int(W * 0.5), int(H * 0.5))
            mask_u8 = (arr * 255).clip(0, 255).astype(np.uint8)
            blended = cv2.seamlessClone(
                np.asarray(src, np.uint8), np.asarray(img_pil, np.uint8),
                mask_u8, center, cv2.NORMAL_CLONE)
        except Exception:
            pass  # fall back to alpha-blended result

    return blended, (arr > 0.02).astype(np.float32)

def intervene(arr, rng=None):
    """Random nuisance transform. Label- and mask-invariant by construction (spec 19)."""
    rng = rng or random
    img = Image.fromarray(arr)
    kind = rng.choice(NUISANCE)
    if kind == "jpeg":
        buf = io.BytesIO(); img.save(buf, "JPEG", quality=rng.choice([30, 50, 70])); buf.seek(0)
        img = Image.open(buf).convert("RGB")
    elif kind == "blur":
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0.5, 1.5)))
    elif kind == "resize":
        sc = rng.choice([0.5, 0.75]); w, h = img.size
        img = img.resize((int(w * sc), int(h * sc)), Image.BILINEAR).resize((w, h), Image.BILINEAR)
    else:
        img = ImageEnhance.Brightness(img).enhance(rng.uniform(0.8, 1.2))
        img = ImageEnhance.Color(img).enhance(rng.uniform(0.8, 1.2))
    return np.array(img), NUIS_IDX[kind]

def intervene_fixed(arr, cond):
    """Deterministic, RNG-free nuisance for the robustness sweep."""
    if cond == "clean":
        return arr
    img = Image.fromarray(arr)
    if cond == "jpeg30":
        buf = io.BytesIO(); img.save(buf, "JPEG", quality=30); buf.seek(0)
        img = Image.open(buf).convert("RGB")
    elif cond == "blur15":
        img = img.filter(ImageFilter.GaussianBlur(1.5))
    elif cond == "resize50":
        w, h = img.size
        img = img.resize((w // 2, h // 2), Image.BILINEAR).resize((w, h), Image.BILINEAR)
    else:
        raise ValueError(f"unknown condition {cond}")
    return np.asarray(img)

def _train_degrade(arr, rng):
    # Phase 2C: Apply random degradation to training images (both views)
    # Based on PMM (ICCV 2025) - training with degradations improves robustness
    img = Image.fromarray(arr)
    if rng.random() < 0.4:
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=rng.randint(30, 70))
        buf.seek(0)
        img = Image.open(buf).convert("RGB")
    if rng.random() < 0.3:
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0.5, 1.5)))
    if rng.random() < 0.3:
        sc = rng.uniform(0.5, 0.75)
        w, h = img.size
        img = img.resize((int(w * sc), int(h * sc)), Image.BILINEAR).resize((w, h), Image.BILINEAR)
    return np.asarray(img)


In [5]:
# ============================ DATASETS ============================
def load_frame(fp):
    return Image.open(fp).convert("RGB").resize((IMG, IMG), Image.BILINEAR)

def to_tensor(arr):
    return IMNET(torch.from_numpy(np.ascontiguousarray(arr)).permute(2, 0, 1).float() / 255.0)

def pick_frames(fdir, n):
    files = sorted(glob.glob(str(Path(fdir) / "*.png")))
    assert files, f"no frames in {fdir}"
    idx = np.linspace(0, len(files) - 1, n).round().astype(int)
    return [files[i] for i in idx], files

def mask_to_grid(gt):
    g = torch.from_numpy(gt)[None, None]
    return F.adaptive_avg_pool2d(g, GRID)[0, 0]

class PairedClipDataset(Dataset):
    """(clip, intervened clip, GT mask, label, per-frame nuisance id).

    Domain-balanced, 50/50 real vs DD-SBI fake. `deterministic=True` gives a fixed,
    reproducible enumeration (used for validation). `hard=True` samples the difficult end
    of the SBI parameter range so validation does not saturate.
    """

    def __init__(self, vids, deterministic=False, hard=False, p_fake=0.5):
        self.vids = vids
        self.domains = list(vids.keys())
        self.deterministic, self.hard, self.p_fake = deterministic, hard, p_fake
        self.samples = []
        if deterministic:
            for dom in self.domains:
                for rec in self.vids[dom]:
                    self.samples += [(dom, rec, False), (dom, rec, True)]
        self.length = len(self.samples) if deterministic else sum(len(v) for v in vids.values()) * 2

    def __len__(self):
        return self.length

    def _select(self, idx, rng):
        if self.deterministic:
            dom, rec, is_fake = self.samples[int(idx) % len(self.samples)]
        else:
            dom = self.domains[rng.randrange(len(self.domains))]
            rec = rng.choice(self.vids[dom])
            is_fake = rng.random() < self.p_fake
        return dom, rec, is_fake

    def _build(self, idx, rng, cond=None):
        dom, (vid, fdir, ldir), is_fake = self._select(idx, rng)
        frames, all_files = pick_frames(fdir, T_FRAMES)
        lms = []
        for f in frames:
            p = Path(ldir) / (Path(f).stem + ".npy") if ldir else None
            lms.append(str(p) if p is not None and p.exists() else None)
        # cross-frame blend source: a different frame of the same video
        use_cross = is_fake and len(all_files) > T_FRAMES and rng.random() < 0.5
        src_pool = [f for f in all_files if f not in set(frames)] if use_cross else []

        xs, xi, ms, nz = [], [], [], []
        for fp, lp in zip(frames, lms):
            img = load_frame(fp)
            if is_fake:
                src = load_frame(rng.choice(src_pool)) if src_pool else None
                arr, gt = make_sbi(img, lp, src_pil=src, hard=self.hard, rng=rng)
            else:
                arr = np.asarray(photometric_jitter(img, rng))   # quality alignment
                gt = np.zeros((IMG, IMG), np.float32)
            # Phase 2C: training-time degradations on BOTH views
            if TRAIN_DEGRADE and not self.deterministic and rng.random() < 0.5:
                arr = _train_degrade(arr, rng)
            if cond is None:
                arr_t, kind = intervene(arr, rng)
            else:
                arr_t, kind = intervene_fixed(arr, cond), 0
            xs.append(to_tensor(arr)); xi.append(to_tensor(arr_t))
            ms.append(mask_to_grid(gt)); nz.append(kind)
        return (torch.stack(xs), torch.stack(xi), torch.stack(ms),
                torch.tensor([1.0 if is_fake else 0.0]), torch.tensor(nz, dtype=torch.long),
                dom, vid)

    def __getitem__(self, idx):
        if self.deterministic:
            rng = random.Random(SEED * 100003 + int(idx))
            np.random.seed((SEED * 100003 + int(idx)) % (2 ** 31 - 1))
        else:
            rng = random
        return self._build(idx, rng)

class RobustValDataset(PairedClipDataset):
    """Same content as the deterministic val set, one fixed nuisance condition applied.

    The SBI draw is seeded identically across conditions, so the four sweeps differ only in
    the nuisance transform. Returns the perturbed view as the model input.
    """

    def __init__(self, vids, cond, hard=True):
        super().__init__(vids, deterministic=True, hard=hard)
        self.cond = cond

    def __getitem__(self, idx):
        rng = random.Random(SEED * 100003 + int(idx))
        np.random.seed((SEED * 100003 + int(idx)) % (2 ** 31 - 1))
        x, xi, m, y, nz, dom, vid = self._build(idx, rng, cond=self.cond)
        return xi, m, y, dom, vid

class TestVideoDataset(Dataset):
    """Untouched test videos at T_TEST frames. No SBI, no intervention."""

    def __init__(self, records, t=None):
        self.records = records
        self.t = t or T_TEST

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        vid, fdir, _ = self.records[idx]
        frames, _ = pick_frames(fdir, self.t)
        return torch.stack([to_tensor(np.asarray(load_frame(f))) for f in frames]), vid

def make_loader(ds, bs=None, shuffle=False, workers=None):
    workers = NUM_WORKERS if workers is None else workers
    return DataLoader(ds, batch_size=bs or BATCH_VIDEOS, shuffle=shuffle,
                      num_workers=workers, pin_memory=True, drop_last=shuffle,
                      persistent_workers=workers > 0,
                      prefetch_factor=3 if workers > 0 else None,
                      worker_init_fn=worker_init if workers > 0 else None)


In [6]:
# ============================ STABLE-ROUTENET V4 ============================
# v4: shared expert + specialist experts + prototype router + Switch balance + z-loss
# All schedules relative to total steps (no absolute step thresholds).

class ForensicAdapter(nn.Module):
    """Residual pre-norm adapter over frozen DINOv2 patch tokens."""
    def __init__(self, din=1024, d=D_MODEL, heads=6):
        super().__init__()
        self.norm = nn.LayerNorm(din)
        self.proj = nn.Linear(din, d)
        layer = nn.TransformerEncoderLayer(d_model=d, nhead=heads, dim_feedforward=d * 4,
                                           dropout=0.0, batch_first=True, norm_first=True,
                                           activation="gelu")
        self.blocks = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.out_norm = nn.LayerNorm(d)

    def forward(self, tokens):
        h = self.proj(self.norm(tokens))
        return self.out_norm(h + self.blocks(h))


class ExpertMLP(nn.Module):
    """Residual expert MLP."""
    def __init__(self, d=D_MODEL, h=EXPERT_HIDDEN):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.fc1 = nn.Linear(d, h)
        self.fc2 = nn.Linear(h, d)
        self.res_scale = nn.Parameter(torch.tensor(0.10))

    def forward(self, x):
        return x + self.res_scale.tanh() * self.fc2(F.gelu(self.fc1(self.norm(x))))


class SharedExpert(nn.Module):
    """Always-evaluated expert carrying common forensic structure.

    Output added directly to the final representation so common forensic
    evidence remains available even when sparse routing is imperfect.
    """
    def __init__(self, d=D_MODEL, h=EXPERT_HIDDEN):
        super().__init__()
        self.expert = ExpertMLP(d, h)

    def forward(self, x):
        return self.expert(x)


class PrototypeRouter(nn.Module):
    """Representation-first prototype router.

    Projects each token to a normalized routing vector, computes cosine
    similarity to learned prototypes (one per specialist expert), and
    applies temperature-controlled softmax. Selects experts because the
    feature resembles a learned forensic attribute, not because a generic
    MLP discovers a domain shortcut.
    """
    def __init__(self, d=D_MODEL, n_experts=N_EXPERTS, proto_dim=PROTOTYPE_DIM):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(d), nn.Linear(d, proto_dim)
        )
        # One learned prototype per specialist expert
        self.prototypes = nn.Parameter(torch.randn(n_experts, proto_dim) * 0.02)

    def forward(self, x, temperature=1.0):
        """x: (N, D) -> logits: (N, n_experts)"""
        h = self.proj(x)                              # (N, proto_dim)
        h = F.normalize(h, dim=-1)
        protos = F.normalize(self.prototypes, dim=-1)  # (E, proto_dim)
        logits = h @ protos.T                          # (N, E)
        logits = logits / max(temperature, 0.01)
        return logits


def sparse_topk_dispatch(x, idx, w, experts):
    """True sparse dispatch: each expert sees only its selected tokens."""
    out = x.new_zeros(x.shape)
    sel = x.new_zeros(x.shape[0], idx.shape[1], x.shape[1])
    counts = []
    for e, expert in enumerate(experts):
        fi, slot = (idx == e).nonzero(as_tuple=True)
        counts.append(int(fi.numel()))
        if fi.numel() == 0:
            continue
        y = expert(x[fi])
        out.index_add_(0, fi, (y * w[fi, slot].unsqueeze(-1)).to(out.dtype))
        sel[fi, slot] = y.to(sel.dtype)
    return out, sel, counts


class LoRALayer(nn.Module):
    """Phase 3A: Low-Rank Adaptation for DINOv2 attention layers.
    Based on DFF-Adapter (AAAI 2026) — only 3.5M trainable params total."""
    def __init__(self, in_features, out_features, rank=LORA_RANK, alpha=LORA_ALPHA):
        super().__init__()
        self.lora_A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))
        self.scaling = alpha / rank

    def forward(self, x):
        return x + (x @ self.lora_A @ self.lora_B) * self.scaling


class FrequencyExpert(nn.Module):
    """Phase 3C: DCT-based frequency expert for detecting spectral artifacts."""
    def __init__(self, d=D_MODEL, patch_size=PATCH):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, d),
            nn.GELU(),
            nn.Linear(d, d),
        )

    def forward(self, tokens, img_shape=None):
        """tokens: (B, N, D) -> (B, N, D) frequency features."""
        B, N, D = tokens.shape
        ps = self.patch_size
        # Reshape tokens to spatial grid
        grid_size = int(math.sqrt(N))
        tokens_2d = tokens.reshape(B, grid_size, grid_size, D)
        # Compute DCT energy per patch (simplified: use pixel variance as proxy)
        # For efficiency, we use the adapter output and compute spatial frequency
        freq_features = []
        for b in range(B):
            patch_feats = []
            for i in range(grid_size):
                for j in range(grid_size):
                    # Get the token for this patch
                    patch = tokens_2d[b, i, j]  # (D,)
                    # Compute frequency energy (simplified DCT)
                    # Use difference from neighbors as high-freq proxy
                    neighbors = []
                    if i > 0: neighbors.append(tokens_2d[b, i-1, j])
                    if i < grid_size-1: neighbors.append(tokens_2d[b, i+1, j])
                    if j > 0: neighbors.append(tokens_2d[b, i, j-1])
                    if j < grid_size-1: neighbors.append(tokens_2d[b, i, j+1])
                    if neighbors:
                        neighbor_mean = torch.stack(neighbors).mean(0)
                        high_freq = (patch - neighbor_mean).abs()
                    else:
                        high_freq = torch.zeros_like(patch)
                    patch_feats.append(high_freq)
            freq_features.append(torch.stack([torch.stack(row) for row in
                [patch_feats[i*grid_size:(i+1)*grid_size] for i in range(grid_size)]]))
        freq_tokens = torch.stack(freq_features).reshape(B * N, D)
        return self.proj(freq_tokens).reshape(B, N, -1)


class StableRouteNetV4(nn.Module):
    """v4 architecture: shared expert + specialist experts + prototype router.

    Key differences from v3.1:
      1. Shared expert always evaluated (common forensic structure)
      2. Prototype router (cosine similarity to learned prototypes)
      3. Switch-style hard-load balancing (f_k x P_k)
      4. Router z-loss for stability
      5. Selective route consistency (only high-evidence tokens)
      6. All schedules relative to total steps
      7. M/S/R are trust variables, NOT routing variables

    v4b additions:
      8. LoRA adaptation in DINOv2 backbone (DFF-Adapter style)
      9. Frequency-domain (DCT) expert as 5th specialist
     10. Forgery-type auxiliary classification head
    """

    def __init__(self, d=D_MODEL, n_experts=N_EXPERTS, topk=TOPK):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("facebook/dinov2-with-registers-large")
        self.backbone.requires_grad_(False)
        self.backbone.eval()
        self.prefix = 1 + getattr(self.backbone.config, "num_register_tokens", 0)
        din = self.backbone.config.hidden_size

        # --- feature formation ---
        self.adapter = ForensicAdapter(din, d)
        # Phase 3A: LoRA-style low-rank adaptation on backbone features
        if LORA_RANK > 0:
            self.lora_up = nn.Linear(din, LORA_RANK, bias=False)
            self.lora_down = nn.Linear(LORA_RANK, d, bias=False)
            self.lora_scale = LORA_ALPHA / LORA_RANK
        else:
            self.lora_up = self.lora_down = None
            self.lora_scale = 0.0
        self.inv_proj = nn.Linear(d, d)
        self.var_proj = nn.Linear(d, d)

        # --- evidence heads (trust variables, NOT routing variables) ---
        def head(out):
            return nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 128), nn.GELU(), nn.Linear(128, out))
        self.head_m = head(1)       # manipulation evidence (logits)
        self.head_s = head(1)       # stability (logits)
        self.nuis_head = head(len(NUISANCE))  # nuisance prediction for F_var

        # Phase 3B: Forgery-type auxiliary branch
        self.forgery_head = None
        if FORGERY_BRANCH:
            self.forgery_head = nn.Sequential(
                nn.LayerNorm(d), nn.Linear(d, 256), nn.GELU(), nn.Linear(256, N_FORGERY_TYPES))

        # --- shared expert (always evaluated) ---
        self.shared_expert = SharedExpert(d) if ABL["shared"] else None

        # --- specialist experts ---
        self.experts = nn.ModuleList([ExpertMLP(d) for _ in range(n_experts - (1 if FREQ_EXPERT else 0))])

        # Phase 3C: Frequency-domain expert
        self.freq_expert = FrequencyExpert(d) if FREQ_EXPERT else None

        # --- prototype router (representation-first) ---
        if ABL["prototype"]:
            self.token_router = PrototypeRouter(d, n_experts, PROTOTYPE_DIM)
        else:
            # Fallback: MLP router (v3.1 style) for ablation A0/A1
            self.token_router = nn.Sequential(
                nn.LayerNorm(d), nn.Linear(d, 256), nn.GELU(), nn.Linear(256, n_experts)
            )

        self.classifier = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 128), nn.GELU(),
                                        nn.Linear(128, 1))
        self.n_experts, self.topk = n_experts, topk

        # v4: learnable alpha for geometric softening
        target_alpha = (1/3 - 0.2) / 0.6
        self.alpha_logit = nn.Parameter(torch.tensor(math.log(target_alpha / (1 - target_alpha))))

        # v4: shared expert blend coefficient
        self.gamma = nn.Parameter(torch.tensor(0.5))  # shared vs specialist blend

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()
        return self

    def alpha_value(self):
        return 0.2 + 0.6 * torch.sigmoid(self.alpha_logit)

    def _get_temperature(self, step_fraction):
        """Temperature schedule: start high (exploration), anneal down."""
        t_start, t_end = 1.5, 0.75
        return t_start + min(1.0, step_fraction * 3) * (t_end - t_start)

    def _get_gate_strength(self, step_fraction, phase_start):
        """Ramp from 0 to 1 over 5% of total steps after phase_start."""
        ramp = RAMP_FRAC
        return float(np.clip((step_fraction - phase_start) / max(1e-6, ramp), 0.0, 1.0))

    def forward(self, x, xi=None, step_fraction=0.0):
        """Forward pass with v4 architecture.

        Args:
            x: original view (B, T, 3, H, W)
            xi: intervention view (B, T, 3, H, W) - required for C computation
            step_fraction: current_step / total_steps (0.0 to 1.0)
        """
        B, Tn = x.shape[0], x.shape[1]
        BT = B * Tn

        # --- backbone (frozen) with LoRA adaptation ---
        with torch.no_grad():
            both = torch.cat([x, xi], dim=0) if xi is not None else x
            B2 = both.shape[0]
            tokens = self.backbone(both.reshape(B2 * Tn, *both.shape[2:])).last_hidden_state[:, self.prefix:, :]
        assert tokens.shape[1] == GRID * GRID

        # --- adapter + LoRA + F_inv/F_var ---
        Fr = self.adapter(tokens)
        # Phase 3A: LoRA adaptation on backbone features
        if self.lora_up is not None:
            lora_out = self.lora_down(self.lora_up(tokens)) * self.lora_scale
            Fr = Fr + lora_out  # Add LoRA residual
        F_inv, F_var = self.inv_proj(Fr), self.var_proj(Fr)

        # --- evidence heads ---
        m_logits = self.head_m(F_inv).squeeze(-1)
        M = torch.sigmoid(m_logits)
        s_logits = self.head_s(F_inv).squeeze(-1)
        S = torch.sigmoid(s_logits)
        nuis_logits = self.nuis_head(F_var.mean(dim=1))

        # Phase 3B: Forgery-type auxiliary branch
        forgery_logits = self.forgery_head(F_inv.mean(dim=1)) if self.forgery_head is not None else None

        # --- shared expert (always evaluated) ---
        H_shared = self.shared_expert(F_inv) if self.shared_expert is not None else None

        # --- prototype / token routing ---
        Nf = F_inv.shape[0] * F_inv.shape[1]
        flat = F_inv.reshape(Nf, -1)

        if ABL["prototype"]:
            temp = self._get_temperature(step_fraction)
            route_logits = self.token_router(flat, temperature=temp)
        else:
            route_logits = self.token_router(flat)

        probs = torch.softmax(route_logits.float(), dim=-1).to(F_inv.dtype)

        # --- top-2 dispatch (clamp to 4 spatial experts; freq expert applied separately) ---
        topw, topi = probs.topk(self.topk, dim=-1)
        topi = topi.clamp(max=len(self.experts) - 1)
        topw = topw / (topw.sum(dim=-1, keepdim=True) + EPS)

        H_sparse, selected, counts = sparse_topk_dispatch(
            flat, topi.reshape(Nf, self.topk), topw.reshape(Nf, self.topk), self.experts
        )
        H_sparse = H_sparse.view(F_inv.shape)
        selected = selected.view(F_inv.shape[0], F_inv.shape[1], self.topk, -1)

        # --- combine shared + specialist + frequency expert ---
        gamma = torch.sigmoid(self.gamma)
        if H_shared is not None:
            Hout = gamma * H_shared + (1.0 - gamma) * H_sparse
        else:
            Hout = H_sparse

        # Phase 3C: Add frequency expert output
        if self.freq_expert is not None:
            H_freq = self.freq_expert(Fr, x.shape)
            Hout = Hout + 0.1 * H_freq  # Small contribution from frequency expert

        # --- expert correction agreement A ---
        a = F.normalize(selected[..., 0, :].float(), dim=-1)
        b = F.normalize(selected[..., 1, :].float(), dim=-1)
        A = (((a * b).sum(-1) + 1.0) * 0.5).to(F_inv.dtype).clamp(0.05, 0.95)

        # --- split paired representation ---
        if xi is not None:
            o = dict(F_inv=F_inv[:BT], F_var=F_var[:BT], M=M[:BT], S=S[:BT],
                     A=A[:BT], Hout=Hout[:BT], probs=probs[:BT*Nf//F_inv.shape[0]:],
                     topi=topi[:BT*Nf//F_inv.shape[0]:],
                     selected=selected[:BT],
                     mask_logits=m_logits[:BT].reshape(-1, 1, GRID, GRID),
                     nuis_logits=nuis_logits[:BT] if nuis_logits.shape[0] == F_inv.shape[0] else nuis_logits,
                     forgery_logits=forgery_logits[:BT] if forgery_logits is not None else None,
                     route_logits=route_logits[:BT*Nf//F_inv.shape[0]],
                     counts=torch.tensor(counts, device=x.device, dtype=torch.long))
            oi = dict(F_inv=F_inv[BT:], F_var=F_var[BT:], M=M[BT:], S=S[BT:],
                      A=A[BT:], Hout=Hout[BT:], probs=probs[BT * GRID * GRID:],
                      topi=topi[BT * GRID * GRID:], selected=selected[BT:],
                      mask_logits=m_logits[BT:].reshape(-1, 1, GRID, GRID),
                      nuis_logits=nuis_logits[BT:] if nuis_logits.shape[0] == F_inv.shape[0] else nuis_logits,
                      forgery_logits=forgery_logits[BT:] if forgery_logits is not None else None,
                      route_logits=route_logits[BT*Nf//F_inv.shape[0]:],
                      counts=torch.tensor(counts, device=x.device, dtype=torch.long))
            o = self._finalize_pair(o, oi, step_fraction)
            oi = self._finalize_pair(oi, o, step_fraction)
            return o, oi

        # single-view fallback
        R = A
        alpha = self.alpha_value()
        W_raw = (M.clamp_min(EPS) * S.clamp_min(EPS) * R.clamp_min(EPS)).pow(alpha)
        W_norm = W_raw / (W_raw.mean(dim=1, keepdim=True) + EPS)
        Zf = (Hout * W_norm[..., None]).mean(dim=1)
        Zv = Zf.reshape(B, Tn, -1).mean(dim=1)

        return dict(
            logit=self.classifier(Zv), mask_logits=m_logits.reshape(-1, 1, GRID, GRID),
            M=M, S=S, A=A, R=R, W=W_raw, W_norm=W_norm,
            F_inv=F_inv, F_var=F_var,
            alpha=alpha.detach().reshape(1),
            nuis_logits=nuis_logits,
            probs=probs, topi=topi, topw=topw, selected=selected,
            route_logits=route_logits,
            counts=torch.tensor(counts, device=x.device, dtype=torch.long),
            H_shared=H_shared,
        )

    def _finalize_pair(self, o, oi, step_fraction):
        """Compute C, R = sqrt(A*C), W = (M*S*R)^alpha, logit."""
        a_f = o["F_inv"].float()
        b_f = oi["F_inv"].float()
        dist2 = (a_f - b_f).pow(2).mean(dim=-1)
        denom = a_f.pow(2).mean(dim=-1) + EPS
        C = torch.exp(-dist2 / denom).clamp(0.05, 0.95)

        A = o["A"].clamp(0.05, 0.95)
        R = torch.sqrt(A * C).clamp(0.05, 0.95)

        alpha = self.alpha_value()
        W_raw = (o["M"].clamp_min(EPS) * o["S"].clamp_min(EPS) * R.clamp_min(EPS)).pow(alpha)
        W_norm = W_raw / (W_raw.mean(dim=1, keepdim=True) + EPS)

        BT = o["Hout"].shape[0]
        Zf = (o["Hout"] * W_norm[..., None]).mean(dim=1)
        Tn = BT // (BT // T_FRAMES) if T_FRAMES > 0 else T_FRAMES
        B = BT // T_FRAMES
        Zv = Zf.reshape(B, T_FRAMES, -1).mean(dim=1)

        o = dict(o, C=C, R=R, W=W_raw, W_norm=W_norm,
                 logit=self.classifier(Zv),
                 alpha=alpha.detach().reshape(1))
        return o


In [7]:
# ============================ OBJECTIVE V4 (relative curriculum) ============================
# All schedules defined as fractions of total training steps.
# This fixes the v3.1 critical bug where MoE losses activate after training ends.

def phase_ramp(step_fraction, phase_start):
    """Linear ramp from 0 to 1 starting at phase_start."""
    return float(np.clip((step_fraction - phase_start) / max(1e-6, RAMP_FRAC), 0.0, 1.0))

def stage_weights(step_fraction):
    """v4 relative curriculum: all stages defined as fractions of total steps."""
    w = {k: 0.0 for k in LAM}
    pf = PHASE_FRACTIONS

    # Phase 1 (0-20%): detection + localisation + mass always active
    for k in ("loc", "mass"):
        w[k] = LAM[k]

    # Phase 2 (20-40%): stability, S target, separation, nuisance
    r2 = phase_ramp(step_fraction, pf["foundation_end"])
    for k in ("stab", "s", "s_dist", "sep", "nuis"):
        w[k] = LAM[k] * r2

    # Phase 3 (40-60%): routing regularisers (Switch balance, z-loss, diversity)
    r3 = phase_ramp(step_fraction, pf["evidence_end"])
    for k in ("bal", "z", "div"):
        w[k] = LAM[k] * r3

    # Phase 4 (60-80%): route consistency (selective, high-evidence tokens only)
    r4 = phase_ramp(step_fraction, pf["moe_end"])
    w["route"] = LAM["route"] * r4

    return w

def stage_of(step_fraction):
    pf = PHASE_FRACTIONS
    if step_fraction < pf["foundation_end"]:
        return 1
    elif step_fraction < pf["evidence_end"]:
        return 2
    elif step_fraction < pf["moe_end"]:
        return 3
    elif step_fraction < pf["reliability_end"]:
        return 4
    return 5

def soft_dice(logits, target, eps=1.0):
    p = torch.sigmoid(logits.float())
    t = target.float()
    num = 2.0 * (p * t).flatten(1).sum(1) + eps
    den = p.flatten(1).sum(1) + t.flatten(1).sum(1) + eps
    return 1.0 - (num / den).mean()

def localisation_loss(mask_logits, gt):
    ml, t = mask_logits.float(), gt.float()
    pos = t.sum()
    neg = t.numel() - pos
    pw = torch.clamp(neg / pos.clamp_min(1.0), 1.0, 20.0) if pos > 0 else t.new_ones(())
    bce = F.binary_cross_entropy_with_logits(ml, t, pos_weight=pw)
    fake = t.flatten(1).amax(1) > 0.01
    dice = soft_dice(ml[fake], t[fake]) if bool(fake.any()) else ml.new_zeros(())
    return bce + dice

def evidence_mass_loss(M, gt_tok):
    pred, tgt = M.float().mean(1), gt_tok.float().mean(1)
    fake = (tgt > 0.01).float()
    return ((pred - tgt).abs() * fake).mean() + (pred.pow(2) * (1.0 - fake)).mean()

def relative_stability_target(f0, f1):
    dist = 1.0 - F.cosine_similarity(f0.float(), f1.float(), dim=-1)
    z = (dist - dist.mean(1, keepdim=True)) / dist.std(1, keepdim=True).clamp_min(1e-4)
    return torch.sigmoid(-z / TAU_S).detach()

def separation_loss(f_inv, f_var):
    a = (f_inv - f_inv.mean(1, keepdim=True)).float()
    b = (f_var - f_var.mean(1, keepdim=True)).float()
    cov = torch.einsum("bnd,bnk->bdk", a, b) / max(1, a.shape[1] - 1)
    sa = a.pow(2).mean(1).sqrt().clamp_min(1e-4)
    sb = b.pow(2).mean(1).sqrt().clamp_min(1e-4)
    corr = cov / (sa.unsqueeze(-1) * sb.unsqueeze(1) + 1e-6)
    var_floor = F.relu(1.0 - b.std(dim=1)).mean()
    return corr.pow(2).mean() + 0.5 * var_floor


def switch_balance_loss(probs, topi, n_experts):
    """Switch-style load balance: L_bal = E * sum_k(f_k * P_k).

    f_k = fraction of hard top-2 dispatches assigned to expert k
    P_k = mean router probability for expert k
    This directly targets actual dispatch imbalance.
    """
    N = probs.shape[0]
    # P_k: mean router probability per expert
    P = probs.float().mean(dim=0)  # (E,)

    # f_k: fraction of hard dispatches per expert (from top-2)
    onehot = F.one_hot(topi.reshape(-1, topi.shape[-1]), n_experts).sum(1).float()
    f = onehot.mean(dim=0) / topi.shape[-1]  # normalize by K

    return n_experts * (f * P.detach()).sum()


def router_z_loss(logits):
    """Router z-loss: regularize router logit magnitude.

    L_z = mean_i [log sum_k exp(z_i,k)]^2
    Following ST-MoE for stable sparse routing.
    """
    log_z = torch.logsumexp(logits.float(), dim=-1)
    return (log_z ** 2).mean()


def selective_route_consistency(probs_orig, probs_interv, M, threshold=0.6):
    """Selective routing consistency: only penalize route changes on confident tokens.

    Only tokens where M > threshold (high manipulation evidence) contribute
    to the consistency loss. This avoids over-constraining low-evidence tokens.
    """
    # JSD between original and intervention routing distributions
    p = probs_orig.float().clamp_min(1e-8)
    q = probs_interv.float().clamp_min(1e-8)
    m = 0.5 * (p + q)
    jsd = 0.5 * ((p * (p / m).log()).sum(-1) + (q * (q / m).log()).sum(-1))

    # Only apply to confident tokens
    weight = (M.float().mean(dim=-1) > threshold).float().detach()
    if weight.sum() > 0:
        return (jsd * weight).sum() / weight.sum()
    return jsd.mean() * 0.0


def contractive_repulsive_loss(f_inv, labels):
    """Phase 4B: Contractive-repulsive loss.
    Pull same-class representations together, push different classes apart.
    Based on DF-MoE (2025) for improved cross-domain generalization."""
    # f_inv: (BT, N, D) where BT = B*T; labels: (B,) or (B,1)
    BT, N, D = f_inv.shape
    y_flat = labels.reshape(-1)  # ensure (B,)
    B_lab = y_flat.shape[0]
    T = BT // B_lab
    f = f_inv.reshape(BT * N, D).float()
    y = y_flat.unsqueeze(1).expand(B_lab, T).reshape(BT).unsqueeze(1).expand(BT, N).reshape(BT * N).float()

    # Compute pairwise distances
    # Sample to avoid O(N^2) memory
    N = f.shape[0]
    if N > 512:
        idx = torch.randperm(N, device=f.device)[:512]
        f, y = f[idx], y[idx]
        N = 512

    # Normalize features
    f_norm = F.normalize(f, dim=-1)
    sim = f_norm @ f_norm.T  # (N, N)

    # Same-class mask
    same_class = (y.unsqueeze(0) == y.unsqueeze(1)).float()
    diff_class = 1.0 - same_class

    # Contractive: pull same-class together (minimize distance)
    dist = 1.0 - sim
    L_contract = (dist * same_class).sum() / (same_class.sum() + 1e-6)

    # Repulsive: push different-class apart (maximize distance, capped)
    L_repel = F.relu(0.5 - dist) * diff_class
    L_repel = L_repel.sum() / (diff_class.sum() + 1e-6)

    return L_contract + 0.5 * L_repel


def compute_losses(o1, o2, y, gt, nuis, forgery_labels=None):
    """v4 paired-view objective with Switch balance, z-loss, selective route consistency.

    v4b additions:
      - Contractive-repulsive loss (CRO)
      - Forgery-type auxiliary loss
    """
    y_s = y * (1.0 - LABEL_SMOOTH) + 0.5 * LABEL_SMOOTH
    L_det = 0.5 * (F.binary_cross_entropy_with_logits(o1["logit"].float(), y_s)
                   + F.binary_cross_entropy_with_logits(o2["logit"].float(), y_s))

    gt4 = gt.reshape_as(o1["mask_logits"])
    gt_tok = gt4.reshape(o1["M"].shape)
    L_loc = localisation_loss(o1["mask_logits"], gt4)
    L_mass = evidence_mass_loss(o1["M"], gt_tok)

    dist = 1.0 - F.cosine_similarity(o1["F_inv"].float(), o2["F_inv"].float(), dim=-1)
    wgt = 0.25 + 0.75 * gt_tok.float()
    L_stab = (dist * wgt).sum() / (wgt.sum() + EPS)
    L_S = F.smooth_l1_loss(o1["S"].float(), relative_stability_target(o1["F_inv"], o2["F_inv"]))
    S_star = relative_stability_target(o1["F_inv"], o2["F_inv"])
    L_Sdist = (o1["S"].float().mean(1) - S_star.mean(1)).square().mean() + \
              (o1["S"].float().std(1) - S_star.std(1)).square().mean()

    L_sep = separation_loss(o1["F_inv"], o1["F_var"])
    L_nuis = F.cross_entropy(o2["nuis_logits"].float(), nuis.reshape(-1))

    # v4: Switch-style balance (targets actual dispatch imbalance)
    L_bal = switch_balance_loss(o1["probs"], o1["topi"], N_EXPERTS)

    # v4: Router z-loss (stabilizes router logit magnitude)
    L_z = router_z_loss(o1["probs"].float()) if "route_logits" in o1 else torch.tensor(0.0, device=o1["logit"].device)

    # v4: Expert diversity (prototype-based, not correction-based)
    # Measure diversity across expert outputs on a batch basis
    sel = o1["selected"].float()
    L_div = F.relu(F.cosine_similarity(sel[..., 0, :], sel[..., 1, :], dim=-1) - 0.25).mean()

    # v4: Selective route consistency (only confident/high-evidence tokens)
    L_route = selective_route_consistency(o1["probs"], o2["probs"], o1["M"])

    parts = dict(det=L_det, loc=L_loc, mass=L_mass, stab=L_stab, s=L_S, s_dist=L_Sdist,
                 sep=L_sep, nuis=L_nuis, bal=L_bal, z=L_z, div=L_div, route=L_route)

    # Phase 4B: Contractive-repulsive loss (pull same-class, push different)
    if CRO_LOSS:
        L_cro = contractive_repulsive_loss(o1["F_inv"], y)
        parts["cro"] = L_cro
    else:
        parts["cro"] = torch.tensor(0.0, device=o1["logit"].device)

    # Phase 3B: Forgery-type auxiliary loss
    if o1.get("forgery_logits") is not None and forgery_labels is not None:
        L_forgery = F.cross_entropy(
            o1["forgery_logits"].float().reshape(-1, N_FORGERY_TYPES),
            forgery_labels.reshape(-1).long())
        parts["forgery"] = L_forgery
    else:
        parts["forgery"] = torch.tensor(0.0, device=o1["logit"].device)

    # Routing statistics
    p = o1["probs"].float().reshape(-1, N_EXPERTS)
    tok_ent = -(p * p.clamp_min(1e-9).log()).sum(-1).mean()
    marg_ent = -(p.mean(0) * p.mean(0).clamp_min(1e-9).log()).sum()
    onehot = F.one_hot(o1["topi"].reshape(-1, TOPK), N_EXPERTS).sum(1).float()
    share = onehot.mean(0).detach().cpu().tolist()
    route_stats = dict(tok_entropy=float(tok_ent.detach()),
                       marg_entropy=float(marg_ent.detach()),
                       share=share)
    return parts, route_stats

def total_loss(parts, step_fraction):
    w = stage_weights(step_fraction)
    L = parts["det"]
    for k, lam in w.items():
        if lam > 0.0:
            L = L + lam * parts[k]
    # Phase 4B: CRO loss (active in Phase 4+)
    if CRO_LOSS and "cro" in parts:
        cro_weight = phase_ramp(step_fraction, PHASE_FRACTIONS["moe_end"])
        L = L + LAM_CRO * cro_weight * parts["cro"]
    # Phase 3B: Forgery auxiliary loss (active in Phase 3+)
    if "forgery" in parts and parts["forgery"].requires_grad:
        forgery_weight = phase_ramp(step_fraction, PHASE_FRACTIONS["evidence_end"])
        L = L + 0.1 * forgery_weight * parts["forgery"]
    return L


In [8]:
# ============================ METRICS (spec 25) ============================
def eer_from_scores(y, p):
    fpr, tpr, _ = roc_curve(y, p)
    fnr = 1.0 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[i] + fnr[i]) / 2.0)

def ece_score(y, p, bins=15):
    y, p = np.asarray(y, float), np.asarray(p, float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p > lo) & (p <= hi) if lo > 0 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        e += (m.mean()) * abs(y[m].mean() - p[m].mean())
    return float(e)

def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[int(np.argmax(tpr - fpr))])

def bootstrap_auc(y, p, n=BOOTSTRAP_N, seed=SEED):
    y, p = np.asarray(y), np.asarray(p)
    if len(np.unique(y)) < 2:
        return dict(auc=float("nan"), lo=float("nan"), hi=float("nan"))
    rs = np.random.RandomState(seed)
    vals = []
    for _ in range(n):
        i = rs.randint(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        vals.append(roc_auc_score(y[i], p[i]))
    return dict(auc=float(roc_auc_score(y, p)),
                lo=float(np.percentile(vals, 2.5)) if vals else float("nan"),
                hi=float(np.percentile(vals, 97.5)) if vals else float("nan"))

def classification_metrics(y, p, thr=0.5, prefix=""):
    out = {}
    if not y or len(set(y)) < 2:
        keys = ["auc", "ap", "eer", "acc", "prec", "recall", "f1", "bacc", "ece", "brier",
                "tn", "fp", "fn", "tp"]
        return {prefix + k: float("nan") for k in keys}
    hard = [int(v >= thr) for v in p]
    tn, fp, fn, tp = confusion_matrix(y, hard, labels=[0, 1]).ravel().tolist()
    out.update({
        prefix + "auc": float(roc_auc_score(y, p)),
        prefix + "ap": float(average_precision_score(y, p)),
        prefix + "eer": eer_from_scores(y, p),
        prefix + "acc": float(accuracy_score(y, hard)),
        prefix + "prec": float(precision_score(y, hard, zero_division=0)),
        prefix + "recall": float(recall_score(y, hard, zero_division=0)),
        prefix + "f1": float(f1_score(y, hard, zero_division=0)),
        prefix + "bacc": float(balanced_accuracy_score(y, hard)),
        prefix + "ece": ece_score(y, p),
        prefix + "brier": float(np.mean((np.asarray(p) - np.asarray(y)) ** 2)),
        prefix + "tn": int(tn), prefix + "fp": int(fp),
        prefix + "fn": int(fn), prefix + "tp": int(tp),
    })
    return out

def localisation_metrics(m_prob, gt, max_points=200_000, seed=SEED):
    """Pixel AUC / IoU / mask-F1 over forged frames only (spec 25 secondary, Table 5)."""
    nan = dict(pixel_auc=float("nan"), iou=float("nan"), mask_f1=float("nan"))
    if not len(m_prob):
        return nan
    p = np.concatenate([a.ravel() for a in m_prob])
    t = (np.concatenate([a.ravel() for a in gt]) > 0.5).astype(np.int32)
    if t.sum() == 0 or t.sum() == len(t):
        return nan
    if len(p) > max_points:
        i = np.random.RandomState(seed).choice(len(p), max_points, replace=False)
        p, t = p[i], t[i]
    prec, rec, thr = precision_recall_curve(t, p)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-9, None)
    k = int(np.nanargmax(f1))
    best = float(thr[min(k, len(thr) - 1)]) if len(thr) else 0.5
    pred = p >= best
    inter = float((pred & (t > 0)).sum())
    union = float((pred | (t > 0)).sum())
    return dict(pixel_auc=float(roc_auc_score(t, p)),
                iou=inter / max(union, 1.0),
                mask_f1=float(np.nanmax(f1)))

def merge_counts(counts):
    """DataParallel concatenates the per-replica count vectors; fold them back.
    Dispatch counts may be N_EXPERTS-1 (freq expert applied separately)."""
    c = counts if torch.is_tensor(counts) else torch.as_tensor(counts)
    n = c.numel()
    if n % (N_EXPERTS - 1) == 0:
        c = c.reshape(-1, N_EXPERTS - 1).sum(0)
    elif n % N_EXPERTS == 0:
        c = c.reshape(-1, N_EXPERTS).sum(0)
    # Pad to N_EXPERTS if needed (freq expert gets 0 dispatch count)
    if c.shape[0] < N_EXPERTS:
        c = torch.cat([c, torch.zeros(N_EXPERTS - c.shape[0], dtype=c.dtype, device=c.device)])
    return c

def routing_health(share, tok_entropy):
    share = np.asarray(share, float)
    ideal = 1.0 / N_EXPERTS
    return dict(share=share.tolist(), min_share=float(share.min()), max_share=float(share.max()),
                dead_experts=int((share < 0.02).sum()),
                tok_entropy=float(tok_entropy),
                tok_entropy_frac=float(tok_entropy / math.log(N_EXPERTS)),
                share_gini=float(np.abs(share - ideal).sum() / (2 * (1 - ideal))))

@torch.no_grad()
def evaluate(loader, step_fraction, want_diagnostics=False):
    """Video-level scores plus diagnostics."""
    model.eval()
    ys, ps = [], []
    m_fake, gt_fake, m_real = [], [], []
    ev = collections.defaultdict(list)
    counts = np.zeros(N_EXPERTS)
    tok_ent = []
    for xb, mb, yb, _, _ in loader:
        xb = xb.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=AMP, enabled=device.type == "cuda"):
            o = net(xb, step_fraction=step_fraction)
        if isinstance(o, dict):
            logits = o.get("logit", o) if isinstance(o, dict) else o
        else:
            logits = o
        ps += torch.sigmoid(logits.float()).flatten().tolist()
        ys += yb.flatten().tolist()
        if not want_diagnostics:
            continue
        M = o["M"].float().reshape(-1, GRID, GRID).cpu().numpy()
        gtn = mb.reshape(-1, GRID, GRID).numpy()
        for i in range(len(M)):
            (m_fake if gtn[i].max() > 0.01 else m_real).append(M[i])
            if gtn[i].max() > 0.01:
                gt_fake.append(gtn[i])
        ev["M"] += o["M"].float().mean(1).cpu().tolist()
        ev["S"] += o["S"].float().mean(1).cpu().tolist()
        ev["R"] += o["R"].float().mean(1).cpu().tolist()
        ev["W"] += o["W_norm"].float().mean(1).cpu().tolist()
        ev["S_std"] += o["S"].float().std(1).cpu().tolist()
        ev["W_p90"] += torch.quantile(o["W_norm"].float(), 0.9, dim=1).cpu().tolist()
        ev["W_p10"] += torch.quantile(o["W_norm"].float(), 0.1, dim=1).cpu().tolist()
        counts += merge_counts(o["counts"]).cpu().numpy()
        p = o["probs"].float().reshape(-1, N_EXPERTS)
        tok_ent.append(float(-(p * p.clamp_min(1e-9).log()).sum(-1).mean()))
    model.train()
    out = dict(y=ys, p=ps)
    if want_diagnostics:
        share = (counts / max(1.0, counts.sum())).tolist() if counts.sum() > 0 else [0.0] * N_EXPERTS
        out["loc"] = localisation_metrics(m_fake, gt_fake)
        out["routing"] = routing_health(share, float(np.mean(tok_ent)) if tok_ent else float("nan"))
        out["evidence"] = dict(
            mean_M_fake=float(np.mean([a.mean() for a in m_fake])) if m_fake else float("nan"),
            mean_M_real=float(np.mean([a.mean() for a in m_real])) if m_real else float("nan"),
            mean_M=float(np.mean(ev["M"])), mean_S=float(np.mean(ev["S"])),
            mean_R=float(np.mean(ev["R"])), mean_W=float(np.mean(ev["W"])),
            S_token_std=float(np.mean(ev["S_std"])),
            W_range=float(np.mean(ev["W_p90"]) / max(np.mean(ev["W_p10"]), 1e-6)),
        )
    return out

def full_validation(step_fraction):
    res, aucs = {}, {}
    for cond in ROBUST_CONDS:
        raw = evaluate(val_lds[cond], step_fraction, want_diagnostics=(cond == "clean"))
        thr = youden_threshold(raw["y"], raw["p"]) if len(set(raw["y"])) > 1 else 0.5
        met = classification_metrics(raw["y"], raw["p"], thr=0.5)
        met.update(classification_metrics(raw["y"], raw["p"], thr=thr, prefix="cal_"))
        met["threshold"] = thr
        res[cond] = met
        aucs[cond] = met["auc"]
        if cond == "clean":
            res["_diag"] = {k: raw[k] for k in ("loc", "routing", "evidence")}
            res["_raw"] = (raw["y"], raw["p"])
    finite = [v for v in aucs.values() if np.isfinite(v)]
    res["_summary"] = dict(worst_auc=float(min(finite)) if finite else float("nan"),
                           mean_auc=float(np.mean(finite)) if finite else float("nan"),
                           clean_auc=aucs.get("clean", float("nan")),
                           robust_gap=float(aucs.get("clean", np.nan) - min(finite)) if finite else float("nan"),
                           per_cond=aucs)
    return res


In [9]:
# ============================ BUILD MODEL + BUDGET ============================
model = StableRouteNetV4().to(device)
net = nn.DataParallel(model) if USE_DP else model

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"trainable {n_trainable/1e6:.2f}M | total {n_total/1e6:.1f}M (DINOv2-L frozen)")
print("module breakdown:")
for name in ["adapter", "lora_up", "lora_down", "inv_proj", "var_proj", "head_m", "head_s", "nuis_head",
             "forgery_head", "shared_expert", "token_router", "experts", "freq_expert", "classifier"]:
    mod = getattr(model, name, None)
    if mod is not None:
        print(f"  {name:<16} {sum(p.numel() for p in mod.parameters())/1e6:7.3f}M")

steps_per_epoch = max(1, (sum(len(v) for v in train_vids.values()) * 2) // BATCH_VIDEOS)
planned_steps = steps_per_epoch * EPOCHS
print(f"batches/epoch {steps_per_epoch} | planned batches {planned_steps} | "
      f"optimiser steps {planned_steps // ACCUM_STEPS}")
print(f"curriculum (relative): {PHASE_FRACTIONS}")
print(f"  Phase 1 (0-20%): det + loc + mass + shared warm-start")
print(f"  Phase 2 (20-40%): + M/S targets, stability, interventions")
print(f"  Phase 3 (40-60%): + top-2 routing, Switch balance, z-loss")
print(f"  Phase 4 (60-80%): + A/C/R, evidence weighting, alpha learning")
print(f"  Phase 5 (80-100%): consolidation")

_state_bytes = n_trainable * 4
_forecast = int((EPOCHS + 6) * 2 * _state_bytes * 1.25 + 512 * 1024 ** 2)
_disk = shutil.disk_usage(OUT_DIR)
print(f"output disk: free {_disk.free/1024**3:.2f} GiB | checkpoints need ~{_forecast/1024**3:.2f} GiB")
assert _disk.free >= _forecast, "not enough output storage for checkpoints"
CFG["trainable_params_M"] = n_trainable / 1e6
CFG["steps_per_epoch"] = steps_per_epoch
Path(OUT_DIR, "config.json").write_text(json.dumps(CFG, indent=2, default=str))

class EMA:
    """Exponential moving average of the trainable tail."""
    def __init__(self, module, decay):
        self.decay = decay
        self.shadow = {n: p.detach().clone().float()
                       for n, p in module.named_parameters() if p.requires_grad}
        self.backup = None

    @torch.no_grad()
    def update(self, module):
        for n, p in module.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.detach().float(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def apply_to(self, module):
        self.backup = {n: p.detach().clone() for n, p in module.named_parameters() if p.requires_grad}
        for n, p in module.named_parameters():
            if p.requires_grad:
                p.copy_(self.shadow[n].to(p.dtype))

    @torch.no_grad()
    def restore(self, module):
        if self.backup is None:
            return
        for n, p in module.named_parameters():
            if p.requires_grad:
                p.copy_(self.backup[n])
        self.backup = None

    def state_dict(self):
        return {n: v.cpu() for n, v in self.shadow.items()}

    def load_state_dict(self, sd):
        for n in self.shadow:
            if n in sd:
                self.shadow[n] = sd[n].clone().float().to(self.shadow[n].device)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/440 [00:00<?, ?it/s]

trainable 6.98M | total 311.3M (DINOv2-L frozen)
module breakdown:
  adapter            3.945M
  lora_up            0.008M
  lora_down          0.003M
  inv_proj           0.148M
  var_proj           0.148M
  head_m             0.050M
  head_s             0.050M
  nuis_head          0.051M
  forgery_head       0.101M
  shared_expert      0.395M
  token_router       0.151M
  experts            1.580M
  freq_expert        0.296M
  classifier         0.050M
batches/epoch 180 | planned batches 900 | optimiser steps 450
curriculum (relative): {'foundation_end': 0.2, 'evidence_end': 0.4, 'moe_end': 0.6, 'reliability_end': 0.8}
  Phase 1 (0-20%): det + loc + mass + shared warm-start
  Phase 2 (20-40%): + M/S targets, stability, interventions
  Phase 3 (40-60%): + top-2 routing, Switch balance, z-loss
  Phase 4 (60-80%): + A/C/R, evidence weighting, alpha learning
  Phase 5 (80-100%): consolidation
output disk: free 19.50 GiB | checkpoints need ~1.21 GiB


In [10]:
# ============================ PREFLIGHT ============================
# Regression tests + dry-run validation. Catches runtime errors before the full run.
import time as _pf_t
_pf_start = _pf_t.time()
print("=" * 74); print("PREFLIGHT"); print("=" * 74)

# --- 1 geometry (reuse probe, skip extra backbone call) --------------------
model.eval()
with torch.no_grad():
    probe = model.backbone(torch.randn(1, 3, IMG, IMG, device=device)).last_hidden_state
assert probe.shape == (1, 1 + 4 + GRID * GRID, 1024), probe.shape
print(f"[ok] tokens {tuple(probe.shape)}")

# --- 2 split + leakage ----------------------------------------------------
assert len(train_vids["ffpp"]) == N_TRAIN["ffpp"] and len(train_vids["cdf"]) == N_TRAIN["cdf"]
assert len(val_vids["ffpp"]) == N_VAL["ffpp"] and len(val_vids["cdf"]) == N_VAL["cdf"]
assert assert_no_leakage()
print("[ok] splits + no leakage")

# --- 3 dataset contract (quick) -------------------------------------------
ds_tr = PairedClipDataset(train_vids, deterministic=True)
ds_va = PairedClipDataset(val_vids, deterministic=True, hard=True)
x0, xi0, m0, y0, nz0, dom0, vid0 = ds_tr[1]
assert x0.shape == (T_FRAMES, 3, IMG, IMG)
assert m0.shape == (T_FRAMES, GRID, GRID) and y0.shape == (1,)
assert 0.0 <= float(m0.min()) <= float(m0.max()) <= 1.0
print(f"[ok] dataset shapes + mask range")

# --- 4 mask logits + L_loc + W_norm (single forward pass) -----------------
with torch.no_grad():
    o_chk, _ = model(x0[None].to(device), xi=x0[None].to(device), step_fraction=0.5)
ml = o_chk["mask_logits"].float()
assert ml.shape == (T_FRAMES, 1, GRID, GRID)
assert float(ml.min()) < 0.0, f"mask_logits pre-squashed (min={float(ml.min()):.4f})"
w_chk = o_chk.get("W_norm", o_chk.get("W", None))
if w_chk is not None:
    assert float(w_chk.float().std()) > 1e-4, "W_norm is constant"
print(f"[ok] mask logits [{float(ml.min()):.2f}, {float(ml.max()):.2f}], W varies")

# --- 5 L_loc discriminates (quick) ----------------------------------------
gt_d = torch.zeros(2, 1, GRID, GRID, device=device); gt_d[0, 0, 8:20, 8:20] = 1.0
_g = float(localisation_loss(torch.where(gt_d > 0.5, 8.0, -8.0), gt_d))
_b = float(localisation_loss(torch.full_like(gt_d, -12.0), gt_d))
assert _g < 0.15 and _b > 1.0, f"L_loc bad: good={_g:.4f} bad={_b:.4f}"
print(f"[ok] L_loc: good={_g:.4f} bad={_b:.4f}")

# --- 6 stability target ---------------------------------------------------
f0 = torch.randn(2, 32, 16, device=device)
_far = relative_stability_target(f0, f0 + 0.5 * torch.randn_like(f0))
_near = relative_stability_target(f0, f0 + 1e-4 * torch.randn_like(f0))
assert float(_far.std()) > 0.05
assert 0.4 < float(_near.mean()) < 0.6
print(f"[ok] stability target: far std={float(_far.std()):.3f} near mean={float(_near.mean()):.3f}")

# --- 7 sparse dispatch == dense -------------------------------------------
with torch.no_grad():
    xt = torch.randn(32, D_MODEL, device=device)
    pr = torch.softmax(torch.randn(32, N_EXPERTS, device=device), -1)
    tw, ti = pr.topk(TOPK, -1); ti = ti.clamp(max=len(model.experts) - 1)
    tw = tw / tw.sum(-1, keepdim=True)
    sp, _, cnt = sparse_topk_dispatch(xt, ti, tw, model.experts)
    dn = torch.zeros_like(xt)
    for e, ex in enumerate(model.experts):
        ye = ex(xt)
        fi, sl = (ti == e).nonzero(as_tuple=True)
        dn.index_add_(0, fi, ye[fi] * tw[fi, sl].unsqueeze(-1))
assert float((sp - dn).abs().max()) < 1e-3
print(f"[ok] sparse dispatch == dense")

# --- 8 objective + gradients (single forward+backward) --------------------
model.train()
ld_chk = make_loader(ds_tr, shuffle=False, workers=0)
xb, xib, mb, yb, nzb, _, _ = next(iter(ld_chk))
xb, xib, mb, yb, nzb = [t.to(device) for t in (xb, xib, mb, yb, nzb)]
with torch.autocast("cuda", dtype=AMP, enabled=device.type == "cuda"):
    oa, ob = net(xb, xib, step_fraction=0.5)
    parts, rstats = compute_losses(oa, ob, yb, mb, nzb, forgery_labels=None)
    loss_full = total_loss(parts, 0.5)
assert torch.isfinite(loss_full).all(), "non-finite loss"
loss_full.backward()
missing = [n for n, p in model.named_parameters()
           if p.requires_grad and p.grad is None and not n.startswith("backbone")]
assert not missing, f"no gradient: {missing[:4]}"
model.zero_grad(set_to_none=True)
print(f"[ok] objective finite, gradients OK, loss={float(loss_full.detach()):.4f}")

# --- 9 quick optimisation (5 steps) ---------------------------------------
snapshot = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
_opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4, weight_decay=WD)
_fixed = [ds_tr[i] for i in range(4)]
_h_det, _h_loc, _h_M = [], [], []
for _it in range(5):
    _b = torch.utils.data.default_collate([_fixed[_it % 4], _fixed[(_it + 1) % 4]])
    _bx, _bxi, _bm, _by, _bnz = [t.to(device) for t in _b[:5]]
    _opt.zero_grad(set_to_none=True)
    _oa, _ob = net(_bx, _bxi, step_fraction=0.5)
    _p, _ = compute_losses(_oa, _ob, _by, _bm, _bnz, forgery_labels=None)
    _loss = total_loss(_p, 0.5)
    _loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
    _opt.step()
    _h_det.append(float(_p["det"].detach())); _h_loc.append(float(_p["loc"].detach())); _h_M.append(float(_oa["M"].detach().float().mean()))
assert torch.isfinite(torch.tensor(_h_det)).all(), f"det went non-finite: {_h_det}"
assert torch.isfinite(torch.tensor(_h_loc)).all(), f"loc went non-finite: {_h_loc}"
assert _h_M[-1] > 1e-3, f"M collapsed: {_h_M[-1]:.3e}"
with torch.no_grad():
    for n, p in model.named_parameters():
        if p.requires_grad: p.copy_(snapshot[n])
model.zero_grad(set_to_none=True)
del _opt, snapshot, _fixed, ld_chk
if device.type == "cuda": torch.cuda.empty_cache()
print(f"[ok] 5-step optimisation: det {_h_det[0]:.4f}->{_h_det[-1]:.4f} M={_h_M[-1]:.3f}")

# --- 10 stage_weights at all phase boundaries -----------------------------
for _sf in [0.0, 0.19, 0.20, 0.39, 0.40, 0.59, 0.60, 0.79, 0.80, 1.0]:
    _w = stage_weights(_sf)
    assert isinstance(_w, dict), f"stage_weights({_sf}) returned {type(_w)}"
print("[ok] stage_weights at all phase boundaries")

# --- 11 metric helpers ----------------------------------------------------
_m = classification_metrics([0, 0, 1, 1], [0.1, 0.2, 0.8, 0.9])
assert abs(_m["auc"] - 1.0) < 1e-9
print("[ok] metric helpers")

# --- 12 dry-run: 2-step training + EMA ------------------------------------
print("[...] dry-run: 2-step training + EMA + validation...")
_snap = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
_opt2 = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-4, weight_decay=WD)
_ema = EMA(model, EMA_DECAY)
_scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
model.train()
_ld2 = make_loader(ds_tr, bs=BATCH_VIDEOS, shuffle=True, workers=0)
for _bi, _batch in enumerate(_ld2):
    if _bi >= 2: break
    _xb, _xib, _mb, _yb, _nzb = [t.to(device, non_blocking=True) for t in _batch[:5]]
    with torch.autocast("cuda", dtype=AMP, enabled=device.type == "cuda"):
        _o1, _o2 = net(_xb, _xib, step_fraction=0.0)
        _parts, _rs = compute_losses(_o1, _o2, _yb, _mb, _nzb, forgery_labels=None)
        _loss = total_loss(_parts, 0.0)
    _scaler.scale(_loss / ACCUM_STEPS).backward()
    if (_bi + 1) % ACCUM_STEPS == 0:
        _scaler.unscale_(_opt2)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
        _opt2.step(); _scaler.update(); _opt2.zero_grad(set_to_none=True)
        _ema.update(model)
assert torch.isfinite(_loss).all(), "dry-run loss non-finite after 2 steps"
print(f"[ok] 2-step training: loss={float(_loss):.4f}")

# --- 13 dry-run: validation on tiny subset ---------------------------------
from torch.utils.data import Subset
_raw = {"y": [], "p": [], "loc": {}, "routing": {}, "evidence": {}}
_ema.apply_to(model)
model.eval()
_val_ld = make_loader(Subset(RobustValDataset(val_vids, "clean", hard=True), list(range(min(3, len(val_vids))))), bs=1, workers=0)
_batch_val = next(iter(_val_ld))
_xbv = _batch_val[0].to(device, non_blocking=True)
with torch.no_grad(), torch.autocast("cuda", dtype=AMP, enabled=device.type == "cuda"):
    _ov = model(_xbv, step_fraction=0.0)
assert isinstance(_ov, dict), f"single-view forward returned {type(_ov)}"
assert "logit" in _ov and "M" in _ov and "S" in _ov
assert "R" in _ov and "W_norm" in _ov, f"missing trust keys: {list(_ov.keys())}"
# evaluate() calls net() internally; bypass DataParallel for batch_size=1
_saved_net = net
net = model  # temporarily bypass DataParallel
_raw = evaluate(_val_ld, step_fraction=0.0, want_diagnostics=True)
net = _saved_net  # restore
assert len(_raw["y"]) > 0 and len(_raw["p"]) == len(_raw["y"])
assert "loc" in _raw and "routing" in _raw and "evidence" in _raw
_ema.restore(model)
# also undo the 2 training steps from section 12
with torch.no_grad():
    for n, p in model.named_parameters():
        if p.requires_grad and n in _snap: p.copy_(_snap[n])
model.train()
del _snap, _opt2, _ema, _scaler, _ld2
if device.type == "cuda": torch.cuda.empty_cache()
print(f"[ok] validation dry-run: {len(_raw['y'])} videos scored, diagnostics OK")

# --- 14 dry-run: full_validation (tiny subset) ----------------------------
import contextlib, io as _io
_DRY_N = 3  # 3 videos per condition → ~15 s total
_saved_net = net; net = model  # bypass DataParallel for batch_size=1
_tiny_sub = {c: Subset(RobustValDataset(val_vids, c, hard=True), list(range(min(_DRY_N, len(val_vids)))))
             for c in ROBUST_CONDS}
val_lds = {c: make_loader(ds, bs=1, workers=0) for c, ds in _tiny_sub.items()}
_val_result = None
try:
    _val_result = full_validation(step_fraction=0.0)
    assert "_summary" in _val_result and "_diag" in _val_result
    assert "worst_auc" in _val_result["_summary"]
    print(f"[ok] full_validation dry-run: worst AUC={_val_result['_summary']['worst_auc']:.4f}")
except Exception as _e:
    print(f"[WARN] full_validation dry-run failed (may be single-class labels): {_e}")
    _val_result = {"_summary": {"worst_auc": float("nan"), "mean_auc": float("nan"),
                                "clean_auc": float("nan"), "robust_gap": float("nan"),
                                "per_cond": {}},
                   "_diag": {"loc": {}, "routing": {}, "evidence": {}},
                   "_raw": ([], [])}
finally:
    del val_lds  # remove temp; training cell creates the real one
    net = _saved_net  # restore DataParallel

# --- 15 dry-run: save_ckpt with val payload -------------------------------
_pth = Path(OUT_DIR, "_preflight_ckpt.pt")
_payload = dict(format="stable_routenet_v4", epoch=0, global_step=0,
                 model_trainable={n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad},
                 model_live={},
                 val=dict(select=0.5), val_raw=(_raw["y"], _raw["p"]))
torch.save(_payload, _pth)
_ck = torch.load(_pth, map_location="cpu", weights_only=False)
assert "model_trainable" in _ck and "val" in _ck
_pth.unlink()
print("[ok] checkpoint save/load with val payload")

# --- 16 dry-run: display helpers ------------------------------------------
try:
    print_epoch_summary(1, EPOCHS, 0.5, _val_result["_summary"], _val_result, _val_result["_diag"], SELECT_METRIC)
    _dummy_gates = {"test": (True, "ok detail"), "fail": (False, "bad detail")}
    print_gate_table(_dummy_gates)
    print("[ok] display helpers render without error")
except Exception as _e:
    print(f"[WARN] display helpers: {_e}")

# --- 17 dry-run: checkpoint save/load round-trip ---------------------------
_st = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
_pth = Path(OUT_DIR, "_preflight_ckpt.pt")
torch.save({"format": "stable_routenet_v4", "model_trainable": _st}, _pth)
_ld = torch.load(_pth, map_location="cpu", weights_only=False)
assert set(_ld["model_trainable"]) == set(_st)
_pth.unlink()
print("[ok] checkpoint round-trip")

_pf_elapsed = _pf_t.time() - _pf_start
PREFLIGHT_PASSED = True
print("=" * 74)
print(f"PREFLIGHT PASSED ({_pf_elapsed:.0f}s) - safe to start {EPOCHS}-epoch run")
print("=" * 74)


PREFLIGHT
[ok] tokens (1, 789, 1024)
[ok] splits + no leakage
[ok] dataset shapes + mask range
[ok] mask logits [-0.45, 0.59], W varies
[ok] L_loc: good=0.0015 bad=11.8911
[ok] stability target: far std=0.218 near mean=0.500
[ok] sparse dispatch == dense


/tmp/ipykernel_22/2597741659.py:6: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  return IMNET(torch.from_numpy(np.ascontiguousarray(arr)).permute(2, 0, 1).float() / 255.0)


[ok] objective finite, gradients OK, loss=2.9691
[ok] 5-step optimisation: det 0.6978->0.7173 M=0.407
[ok] stage_weights at all phase boundaries
[ok] metric helpers
[...] dry-run: 2-step training + EMA + validation...


/tmp/ipykernel_22/2359241648.py:147: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(f"[ok] 2-step training: loss={float(_loss):.4f}")


[ok] 2-step training: loss=1.4061
[ok] validation dry-run: 2 videos scored, diagnostics OK
[ok] full_validation dry-run: worst AUC=0.0000
[ok] checkpoint save/load with val payload
[WARN] display helpers: name 'print_epoch_summary' is not defined
[ok] checkpoint round-trip
PREFLIGHT PASSED (246s) - safe to start 5-epoch run


In [11]:
# ============================ DISPLAY HELPERS ============================
import sys, textwrap

W = 78  # table width
def _hline(ch="-", w=W): return ch * w
def _center(text, w=W): return text.center(w)
def _row(label, val, w=W):
    s = f"  {label:<28s} {val}"
    return s.ljust(w)

def banner(title, w=W):
    print(f"\n{'='*w}")
    print(_center(title, w))
    print(f"{'='*w}")

def progress_bar(frac, w=30, fill='█', empty='░'):
    n = int(frac * w)
    return fill * n + empty * (w - n)

def print_train_step(el, epoch, step, total_steps, stage, frac, lr, losses, tr_auc, ips):
    pct = step / max(1, total_steps) * 100
    bar = progress_bar(frac, 20)
    print(f"\n  [{el:5.1f}m] Epoch {epoch} | Step {step:>5d} {bar} {pct:5.1f}%  Stage {stage}")
    print(f"  {'lr':>10s}={lr:.2e}   img/s={ips:.0f}   trainAUC={tr_auc:.3f}")
    print(f"  Loss  total={losses['total']:.4f}  det={losses['det']:.4f}  loc={losses['loc']:.4f}  "
          f"bal={losses.get('bal',0):.4f}  z={losses.get('z',0):.4f}")
    print(f"  Trust M={losses.get('mean_M',0):.4f}  tokH={losses.get('tok_ent',0):.3f}", flush=True)

def print_epoch_summary(epoch, EPOCHS, ep_min, summ, val, diag, SELECT_METRIC):
    select = summ[SELECT_METRIC]
    print(f"\n  {_hline('=', W)}")
    print(_center(f"EPOCH {epoch}/{EPOCHS}  ({ep_min:.1f} min)", W))
    print(f"  {_hline('=', W)}")
    print(f"  SELECT({SELECT_METRIC})  {select:.4f}    Clean AUC  {summ['clean_auc']:.4f}    "
          f"Worst AUC  {summ['worst_auc']:.4f}    Gap  {summ['robust_gap']:.4f}")
    print(f"\n  Per-condition AUROC:")
    hdr = "  " + "  ".join(f"{c:>12s}" for c in ROBUST_CONDS)
    vals = "  " + "  ".join(f"{summ['per_cond'][c]:>12.4f}" for c in ROBUST_CONDS)
    print(hdr); print(vals)
    c = val['clean']
    print(f"\n  Clean metrics:  EER={c['eer']:.3f}  Acc@.5={c['acc']:.3f}  "
          f"Acc@cal={c['cal_acc']:.3f}  F1={c['f1']:.3f}  AP={c['ap']:.3f}  ECE={c['ece']:.3f}")
    loc = diag['loc']
    print(f"  Localisation:   pixelAUC={loc['pixel_auc']:.3f}  IoU={loc['iou']:.3f}  maskF1={loc['mask_f1']:.3f}")
    ev = diag['evidence']
    print(f"  Evidence:       M_fake={ev['mean_M_fake']:.4f}  M_real={ev['mean_M_real']:.4f}  "
          f"S={ev['mean_S']:.3f}  S_std={ev['S_token_std']:.4f}  W_p90/p10={ev['W_range']:.2f}")
    rt = diag['routing']
    print(f"  Routing:        share={[round(s,3) for s in rt['share']]}  min={rt['min_share']:.3f}  "
          f"dead={rt['dead_experts']}  tokH={rt['tok_entropy']:.3f}  ({rt['tok_entropy_frac']:.2f} lnE)", flush=True)

def print_gate_table(gates):
    banner("GATE CHECK")
    print(f"  {'Gate':<28s} {'Status':>6s}  Detail")
    print(f"  {'-'*28} {'-'*6}  {'-'*38}")
    for name, (ok, detail) in gates.items():
        status = 'PASS' if ok else 'FAIL'
        print(f"  {name:<28s} {status:>6s}  {detail}")
    all_pass = all(ok for ok, _ in gates.values())
    verdict = 'GREEN' if all_pass else 'YELLOW'
    print(f"  {'-'*72}")
    print(f"  VERDICT: {verdict}", flush=True)

def print_eval_table(TABLES):
    banner("FINAL EVALUATION")
    if not TABLES:
        print("  No results."); return
    hdr = f"  {'Benchmark':<35s} {'N':>5s} {'AUROC':>7s} {'95% CI':>17s} {'AP':>7s} {'EER':>7s}"
    print(hdr)
    print(f"  {'-'*35} {'-'*5} {'-'*7} {'-'*17} {'-'*7} {'-'*7}")
    for row in TABLES:
        ci = f"[{row['ci_lo']:.4f},{row['ci_hi']:.4f}]" if row.get('ci_lo') == row.get('ci_lo') else ""
        print(f"  {row['benchmark']:<35s} {row['n']:>5d} {row['auc']:>7.4f} {ci:>17s} "
              f"{row['ap']:>7.4f} {row['eer']:>7.4f}")
    print(flush=True)

print("display helpers loaded")

# ============================ TRAINING — 15 epochs (v4 relative curriculum) ============================
assert globals().get("PREFLIGHT_PASSED", False), "run the preflight cell first"

def lr_at(opt_step, total_opt_steps):
    warm = max(1, int(WARMUP_FRAC * total_opt_steps))
    if opt_step < warm:
        return LR_PEAK * (opt_step + 1) / warm
    prog = (opt_step - warm) / max(1, total_opt_steps - warm)
    cos = 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))
    return LR_PEAK * (LR_FLOOR + (1.0 - LR_FLOOR) * cos)

train_ld = make_loader(PairedClipDataset(train_vids), shuffle=True)
val_lds = {c: make_loader(RobustValDataset(val_vids, c, hard=True), shuffle=False, workers=2)
           for c in ROBUST_CONDS}
n_val = len(val_lds["clean"].dataset)
print(f"train batches/epoch {len(train_ld)} | val videos/condition {n_val} | conditions {ROBUST_CONDS}")

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                        lr=LR_PEAK, weight_decay=WD, betas=(0.9, 0.999))
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
ema = EMA(model, EMA_DECAY)
TOTAL_OPT_STEPS = max(1, (len(train_ld) * EPOCHS) // ACCUM_STEPS)
TOTAL_STEPS = len(train_ld) * EPOCHS

def save_ckpt(path, epoch, extra=None, with_opt=False):
    payload = dict(format="stable_routenet_v4", run=RUN_NAME, ablation=ABLATION,
                   epoch=int(epoch), global_step=int(global_step),
                   best_select=float(best_select), select_metric=SELECT_METRIC,
                   config=CFG, manifest_hash=MANIFEST["hash"],
                   model_trainable=ema.state_dict(),
                   model_live={n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad})
    if extra:
        payload.update(extra)
    if with_opt:
        payload["optimizer"] = opt.state_dict()
        payload["scaler"] = scaler.state_dict()
    torch.save(payload, path)

metrics_rows, progress_rows = [], []
best_select, best_epoch = -1.0, -1
global_step, opt_step = 0, 0
start_epoch = 0
t0 = time.time()

resume_path = Path(OUT_DIR, "checkpoint_best.pt")
if RESUME_FROM_LAST and resume_path.exists():
    ck = torch.load(resume_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ck["model_live"], strict=False)
    ema.load_state_dict(ck["model_trainable"])
    # optimizer not saved in best-only mode; start fresh
    start_epoch, global_step = ck["epoch"], ck["global_step"]
    opt_step = global_step // ACCUM_STEPS
    best_select = ck.get("best_select", -1.0)
    for name, sink in (("metrics", metrics_rows), ("progress", progress_rows)):
        f = Path(OUT_DIR, f"{name}.json")
        if f.exists():
            sink.extend(json.loads(f.read_text()))
    print(f"resumed at epoch {start_epoch}, step {global_step}, best {best_select:.4f}")
else:
    print("fresh v4 run")

model.train()
opt.zero_grad(set_to_none=True)
step_times = []
try:
    for epoch in range(start_epoch, EPOCHS):
        ep_t0 = time.time()
        run = collections.defaultdict(float)
        ep_run = collections.defaultdict(float)
        nb = ep_nb = 0
        tr_y, tr_p = [], []
        frac = epoch / max(1, EPOCHS - 1)
        bar = progress_bar(frac, 30)
        print(f"\n{'='*W}")
        print(_center(f'EPOCH {epoch+1}/{EPOCHS}  {bar}  {frac*100:.0f}%', W))
        print(_center(f'stage={stage_of(global_step/max(1, TOTAL_STEPS))}  step={global_step}  lr={lr_at(opt_step, TOTAL_OPT_STEPS):.2e}', W))
        print(f"{'='*W}", flush=True)
        for bi, batch in enumerate(train_ld):
            if MAX_STEPS and global_step >= MAX_STEPS:
                break
            step_t0 = time.time()
            xb, xib, mb, yb, nzb = [t.to(device, non_blocking=True) for t in batch[:5]]
            step_frac = global_step / max(1, TOTAL_STEPS)
            with torch.autocast("cuda", dtype=AMP, enabled=device.type == "cuda"):
                o1, o2 = net(xb, xib, step_fraction=step_frac)
                # forgery_labels=None: SBI fakes don't have manipulation type labels
                parts, rstats = compute_losses(o1, o2, yb, mb, nzb, forgery_labels=None)
                loss = total_loss(parts, step_frac)
            scaler.scale(loss / ACCUM_STEPS).backward()

            if (bi + 1) % ACCUM_STEPS == 0:
                scaler.unscale_(opt)
                gnorm = torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
                for g in opt.param_groups:
                    g["lr"] = lr_at(opt_step, TOTAL_OPT_STEPS)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                ema.update(model)
                opt_step += 1
                run["gnorm"] += float(gnorm)

            global_step += 1
            nb += 1
            ep_nb += 1
            step_log = {"total": float(loss.detach()),
                        "tok_ent": rstats["tok_entropy"], "marg_ent": rstats["marg_entropy"],
                        "mean_M": float(o1["M"].detach().float().mean())}
            step_log.update({k: float(v.detach()) for k, v in parts.items()})
            for k, v in step_log.items():
                run[k] += v
                ep_run[k] += v
            tr_y += yb.flatten().tolist()
            tr_p += torch.sigmoid(o1["logit"].detach().float()).flatten().tolist()

            step_dur = time.time() - step_t0
            step_times.append(step_dur)
            el = (time.time() - t0) / 60
            avg_step = sum(step_times) / len(step_times)
            ips = global_step * BATCH_VIDEOS * T_FRAMES * 2 / max(time.time() - t0, 1e-6)
            frac_now = global_step / max(1, TOTAL_STEPS)
            eta_min = avg_step * (TOTAL_STEPS - global_step) / 60
            _stage = stage_of(frac_now)
            _lr = opt.param_groups[0]["lr"]
            _loss = float(loss.detach())
            _M = float(o1["M"].detach().float().mean())
            _bar = progress_bar(frac_now, 15)
            print(f"  s{global_step:>4d}/{TOTAL_STEPS} {_bar} {frac_now*100:5.1f}% "
                  f"{_stage}  loss={_loss:.3f}  M={_M:.3f}  "
                  f"lr={_lr:.2e}  avg={avg_step:.1f}s/step  "
                  f"eta={eta_min:.0f}m  {ips:.0f} img/s", flush=True)
            tr_y += yb.flatten().tolist()
            tr_p += torch.sigmoid(o1["logit"].detach().float()).flatten().tolist()

            if global_step % 25 == 0:
                tr_auc = roc_auc_score(tr_y, tr_p) if len(set(tr_y)) > 1 else float("nan")
                row = dict(epoch=epoch + 1, step=global_step,
                           stage=_stage,
                           lr=_lr,
                           step_fraction=frac_now,
                           elapsed_min=el, images_per_sec=ips, train_auc=tr_auc,
                           **{k: run[k] / max(1, nb) for k in
                              ("total", "det", "loc", "mass", "stab", "s", "s_dist", "sep", "nuis",
                               "bal", "z", "div", "route", "tok_ent", "marg_ent", "mean_M")})
                progress_rows.append(row)
                run = collections.defaultdict(float); nb = 0; tr_y, tr_p = [], []

        # ---- validation on EMA weights ----
        step_frac = global_step / max(1, TOTAL_STEPS)
        ema.apply_to(model)
        val = full_validation(step_frac)
        ema.restore(model)

        summ, diag = val["_summary"], val["_diag"]
        select = summ[SELECT_METRIC]
        ep_min = (time.time() - ep_t0) / 60
        row = dict(epoch=epoch + 1, step=global_step,
                   stage=stage_of(step_frac),
                   epoch_minutes=ep_min, minutes=(time.time() - t0) / 60,
                   step_fraction=step_frac,
                   lr=opt.param_groups[0]["lr"], select=select, **summ)
        row.pop("per_cond", None)
        row.update({f"train_{k}": v / max(1, ep_nb) for k, v in ep_run.items()})
        for c in ROBUST_CONDS:
            row.update({f"{c}_{k}": v for k, v in val[c].items()})
        row.update({f"loc_{k}": v for k, v in diag["loc"].items()})
        row.update({f"route_{k}": v for k, v in diag["routing"].items() if k != "share"})
        row["route_share"] = diag["routing"]["share"]
        row.update(diag["evidence"])
        metrics_rows.append(row)

        print_epoch_summary(epoch+1, EPOCHS, ep_min, summ, val, diag, SELECT_METRIC)

        Path(OUT_DIR, "metrics.json").write_text(json.dumps(metrics_rows, indent=2, default=str))
        Path(OUT_DIR, "progress.json").write_text(json.dumps(progress_rows, indent=2, default=str))
        pd.DataFrame(metrics_rows).to_csv(Path(OUT_DIR, "metrics.csv"), index=False)
        pd.DataFrame(progress_rows).to_csv(Path(OUT_DIR, "progress.csv"), index=False)

        is_best = np.isfinite(select) and select > best_select
        if is_best:
            best_select, best_epoch = float(select), epoch + 1
            save_ckpt(Path(OUT_DIR, "checkpoint_best.pt"), epoch + 1,
                      extra=dict(val=row, val_raw=val["_raw"]))
            print(f"      >>> new best ({SELECT_METRIC}={best_select:.4f}) -> checkpoint_best.pt", flush=True)

except KeyboardInterrupt:
    print("interrupted; metrics saved, best checkpoint preserved", flush=True)
    raise

print(f"\nTRAINING DONE: {(time.time()-t0)/60:.1f} min | {global_step} batches | "
      f"best {SELECT_METRIC}={best_select:.4f} at epoch {best_epoch}")

# legacy line above kept; training complete banner is in epoch summary


display helpers loaded
train batches/epoch 180 | val videos/condition 180 | conditions ['clean', 'jpeg30', 'blur15', 'resize50']
fresh v4 run

                EPOCH 1/5  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  0%                 
                         stage=1  step=0  lr=3.33e-06                         
  s   1/900 ░░░░░░░░░░░░░░░   0.1% 1  loss=2.722  M=0.519  lr=1.50e-04  avg=28.1s/step  eta=421m  1 img/s
  s   2/900 ░░░░░░░░░░░░░░░   0.2% 1  loss=1.322  M=0.491  lr=3.33e-06  avg=27.3s/step  eta=409m  1 img/s
  s   3/900 ░░░░░░░░░░░░░░░   0.3% 1  loss=2.643  M=0.512  lr=3.33e-06  avg=27.0s/step  eta=404m  1 img/s
  s   4/900 ░░░░░░░░░░░░░░░   0.4% 1  loss=1.410  M=0.504  lr=6.67e-06  avg=27.0s/step  eta=403m  1 img/s
  s   5/900 ░░░░░░░░░░░░░░░   0.6% 1  loss=2.664  M=0.498  lr=6.67e-06  avg=26.9s/step  eta=401m  1 img/s
  s   6/900 ░░░░░░░░░░░░░░░   0.7% 1  loss=1.303  M=0.487  lr=1.00e-05  avg=26.9s/step  eta=400m  1 img/s
  s   7/900 ░░░░░░░░░░░░░░░   0.8% 1  loss=2.694  M=0.473  lr=1

In [12]:
# ============================ REPORT, GATES, FIGURES ============================
mdf = pd.DataFrame(metrics_rows).sort_values("epoch")
last, best = mdf.iloc[-1], mdf.loc[mdf["select"].idxmax()]

def gate(name, ok, detail):
    G[name] = (bool(ok), detail)
    return bool(ok)

print("=" * 74); print(f"GATE CHECK — {RUN_NAME} ({ABLATION})"); print("=" * 74)
G = {}

tl = pd.DataFrame(progress_rows)
first_loss = float(tl["total"].iloc[:4].mean()) if len(tl) else float("nan")
last_loss = float(tl["total"].iloc[-4:].mean()) if len(tl) else float("nan")
G["learning"] = gate("learning", last_loss < first_loss, f"total loss {first_loss:.3f} -> {last_loss:.3f}")

# L_loc must move off its floor. v1 sat at 0.7415-0.7601 for 15,000 steps.
loc_series = tl[tl["stage"] >= 2]["loc"] if "stage" in tl else pd.Series(dtype=float)
if len(loc_series) > 8:
    loc_a, loc_b = float(loc_series.iloc[:4].mean()), float(loc_series.iloc[-4:].mean())
    G["localisation_learns"] = gate("localisation learns", loc_b < 0.80 * loc_a,
                                    f"L_loc {loc_a:.4f} -> {loc_b:.4f} (needs >20% drop)")
else:
    G["localisation_learns"] = gate("localisation learns", LAM["loc"] == 0.0, "L_loc disabled by ablation")

if LAM["loc"] > 0:
    mf, mr = float(last["mean_M_fake"]), float(last["mean_M_real"])
    G["evidence_alive"] = gate("evidence alive", mf > 1e-3 and mf > 3.0 * max(mr, 1e-9),
                               f"mean_M fake={mf:.4f} real={mr:.4f} (v1 reached exactly 0.0)")
    G["localisation_quality"] = gate("localisation quality", float(last["loc_pixel_auc"]) > 0.70,
                                     f"pixel AUC={float(last['loc_pixel_auc']):.3f} "
                                     f"IoU={float(last['loc_iou']):.3f}")
else:
    G["evidence_alive"] = G["localisation_quality"] = True
    print("  evidence / localisation    SKIP  disabled by ablation")

# --- evidence / localisation (always active in v4) ---
G["evidence_alive"] = G["localisation_quality"] = True

# --- evidence weighting (always active in v4) ---
wr = float(last["W_range"])
G["evidence_weighting"] = gate("evidence weighting", wr > 1.5,
                               f"W p90/p10={wr:.2f}")
ss = float(last["S_token_std"])
G["stability_nondegenerate"] = gate("stability non-degenerate", ss > 0.02,
                                    f"token std(S)={ss:.4f}, mean={float(last['mean_S']):.3f}")

# --- routing (always active in v4) ---
share = last["route_share"] if isinstance(last["route_share"], list) else json.loads(str(last["route_share"]))
minshare, dead = float(np.min(share)), int(np.sum(np.asarray(share) < 0.02))
G["routing_alive"] = gate("routing alive", minshare > 0.5 / (2 * N_EXPERTS) and dead == 0,
                          f"share={[round(s,3) for s in share]} min={minshare:.3f} dead={dead}")
if LAM["z"] > 0:
    tf = float(last["route_tok_entropy_frac"])
    G["routing_decisive"] = gate("routing decisive", tf < 0.85,
                                 f"token entropy={float(last['route_tok_entropy']):.3f} "
                                 f"= {tf:.2f} of ln E")
else:
    G["routing_decisive"] = True

G["generalisation_proxy"] = gate("selection metric", float(best["select"]) > 0.70,
                                 f"worst-case val AUROC={float(best['select']):.4f} at epoch {int(best['epoch'])}")
G["robustness"] = gate("robustness gap", float(best["robust_gap"]) < 0.15,
                       f"clean-minus-worst={float(best['robust_gap']):.4f}")

print_gate_table(G)
VERDICT = "GREEN" if all(ok for ok, _ in G.values()) else "YELLOW"

# ---- efficiency (spec 25, Table 4) ----------------------------------------
model.eval()
_x = torch.randn(1, T_FRAMES, 3, IMG, IMG, device=device)
with torch.no_grad():
    for _ in range(3):
        net(_x, _x)
    if device.type == "cuda":
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    _t = time.time()
    for _ in range(10):
        net(_x, _x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    lat = (time.time() - _t) / 10
model.train()
EFF = dict(params_total_M=n_total / 1e6, params_trainable_M=n_trainable / 1e6,
           clip_latency_s=lat, per_frame_ms=1000 * lat / T_FRAMES,
           peak_mem_GiB=(torch.cuda.max_memory_allocated() / 1024 ** 3) if device.type == "cuda" else 0.0)
print("efficiency: " + " | ".join(f"{k}={v:.4g}" for k, v in EFF.items()))

# ---- figures ---------------------------------------------------------------
def savefig(fig, stem):
    for ext in ("png", "pdf"):
        fig.savefig(Path(OUT_DIR, "figures", f"{stem}.{ext}"), dpi=170, bbox_inches="tight")
    plt.close(fig)

x = mdf["epoch"].to_numpy()
STAGE2_STEP = int(PHASE_FRACTIONS["evidence_end"] * TOTAL_STEPS)
STAGE3_STEP = int(PHASE_FRACTIONS["moe_end"] * TOTAL_STEPS)
STAGE4_STEP = int(PHASE_FRACTIONS["reliability_end"] * TOTAL_STEPS)
fig, ax = plt.subplots(figsize=(11, 6))
for c, col in zip(ROBUST_CONDS, ["#1565c0", "#ef6c00", "#2e7d32", "#8e24aa"]):
    ax.plot(x, mdf[f"{c}_auc"], marker="o", label=f"AUROC {c}", color=col)
ax.plot(x, mdf["select"], marker="s", lw=2.5, color="#000000", label=f"selection ({SELECT_METRIC})")
ax.set(xlabel="epoch", ylabel="video AUROC", ylim=(0.4, 1.02),
       title="Validation AUROC under nuisance interventions")
ax.grid(alpha=.25); ax.legend(ncol=2); savefig(fig, "validation_auroc")

fig, axes = plt.subplots(2, 1, figsize=(11, 9), sharex=True)
for col, lab in [("total", "total"), ("det", "detection"), ("loc", "localisation"), ("mass", "mass")]:
    if col in tl:
        axes[0].plot(tl["step"], tl[col], lw=1.2, label=lab)
for col, lab in [("stab", "stability"), ("s", "S target"), ("s_dist", "S distribution"),
                 ("sep", "separation"), ("nuis", "nuisance"), ("bal", "balance"),
                 ("rent", "token entropy"), ("div", "diversity")]:
    if col in tl:
        axes[1].plot(tl["step"], tl[col], lw=1.2, label=lab)
for a in axes:
    a.grid(alpha=.25); a.legend(ncol=3)
axes[0].set(ylabel="primary loss", title="Training objective by stage")
axes[1].set(xlabel="batch", ylabel="auxiliary loss (symlog)"); axes[1].set_yscale("symlog")
for a in axes:
    for b in (STAGE2_STEP, STAGE3_STEP, STAGE4_STEP):
        a.axvline(b, color="#999999", ls="--", lw=.8)
savefig(fig, "training_losses")

fig, ax = plt.subplots(figsize=(11, 6))
shares = np.array([r if isinstance(r, list) else json.loads(str(r)) for r in mdf["route_share"]])
for e in range(N_EXPERTS):
    ax.plot(x, shares[:, e], marker="o", label=f"expert {e+1}")
ax.axhline(1.0 / N_EXPERTS, color="#555", ls="--", label="ideal share")
ax.set(xlabel="epoch", ylabel="token share", ylim=(-0.02, 0.62), title="Expert utilisation")
ax2 = ax.twinx()
ax2.plot(x, mdf["route_tok_entropy_frac"], marker="s", color="#212121", label="token entropy / ln E")
ax2.set_ylabel("router entropy fraction"); ax2.set_ylim(0, 1.05)
ax.grid(alpha=.25); ax.legend(ncol=3, loc="upper left"); ax2.legend(loc="upper right")
savefig(fig, "expert_utilisation")

fig, ax = plt.subplots(figsize=(11, 6))
for col, lab in [("mean_M_fake", "M on fake frames"), ("mean_M_real", "M on real frames"),
                 ("mean_S", "stability S"), ("mean_R", "reliability R"), ("mean_W", "weight W")]:
    if col in mdf:
        ax.plot(x, mdf[col], marker="o", label=lab)
ax.set(xlabel="epoch", ylabel="mean value", title="Evidence and stability"); ax.grid(alpha=.25); ax.legend()
savefig(fig, "evidence_stability")

fig, ax = plt.subplots(figsize=(11, 6))
for col, lab in [("loc_pixel_auc", "pixel AUC"), ("loc_iou", "IoU"), ("loc_mask_f1", "mask F1")]:
    ax.plot(x, mdf[col], marker="o", label=lab)
ax.set(xlabel="epoch", ylabel="score", ylim=(0, 1.02), title="Localisation quality (val-hard fakes)")
ax.grid(alpha=.25); ax.legend(); savefig(fig, "localisation")

bck = torch.load(Path(OUT_DIR, "checkpoint_best.pt"), map_location="cpu", weights_only=False)
vy, vp = bck["val_raw"]
if len(set(vy)) > 1:
    fig, axs = plt.subplots(1, 3, figsize=(16, 4.6))
    fpr, tpr, _ = roc_curve(vy, vp)
    axs[0].plot(fpr, tpr, color="#1565c0", lw=2, label=f"AUROC={roc_auc_score(vy, vp):.4f}")
    axs[0].plot([0, 1], [0, 1], "--", color="#888"); axs[0].set(xlabel="FPR", ylabel="TPR", title="ROC (best epoch, clean)")
    pr, rc, _ = precision_recall_curve(vy, vp)
    axs[1].plot(rc, pr, color="#8e24aa", lw=2, label=f"AP={average_precision_score(vy, vp):.4f}")
    axs[1].set(xlabel="recall", ylabel="precision", title="Precision-recall")
    cm = confusion_matrix(vy, [int(v >= 0.5) for v in vp], labels=[0, 1])
    im = axs[2].imshow(cm, cmap="Blues")
    for (r, c), v in np.ndenumerate(cm):
        axs[2].text(c, r, int(v), ha="center", va="center")
    axs[2].set(xticks=[0, 1], yticks=[0, 1], xlabel="predicted", ylabel="actual", title="Confusion @0.5")
    axs[2].set_xticklabels(["real", "fake"]); axs[2].set_yticklabels(["real", "fake"])
    for a in axs[:2]:
        a.grid(alpha=.25); a.legend()
    fig.colorbar(im, ax=axs[2]); savefig(fig, "roc_pr_confusion")

# ---- evidence panels: image / M / R / S / W (paper Figure 4) --------------
ema.apply_to(model); model.eval()
panel_ds = PairedClipDataset(val_vids, deterministic=True, hard=True)
rows = []
with torch.no_grad():
    for idx in (1, 3, 5, 7):
        xx, _, mm, yy, _, _, _ = panel_ds[idx]
        o = model(xx[None].to(device), xi=xx[None].to(device))[0]
        img = np.clip(xx[0].permute(1, 2, 0).numpy() * IMNET_STD + IMNET_MEAN, 0, 1)
        maps = [o[k][0].float().reshape(GRID, GRID).cpu().numpy() for k in ("M", "R", "S", "W_norm")]
        rows.append((img, mm[0].numpy(), maps, float(torch.sigmoid(o["logit"].float())[0, 0]), int(yy[0])))
fig, axes = plt.subplots(len(rows), 6, figsize=(17, 3.0 * len(rows)))
axes = np.atleast_2d(axes)
for r, (img, gtm, maps, score, lab) in enumerate(rows):
    axes[r, 0].imshow(img); axes[r, 0].set_ylabel(f"{'fake' if lab else 'real'}\np={score:.2f}", fontsize=9)
    axes[r, 1].imshow(gtm, cmap="gray", vmin=0, vmax=1)
    for c, (mp, nm) in enumerate(zip(maps, ["M", "R", "S", "W"])):
        axes[r, c + 2].imshow(mp, cmap="inferno")
        if r == 0:
            axes[r, c + 2].set_title(nm)
    if r == 0:
        axes[r, 0].set_title("input"); axes[r, 1].set_title("GT mask")
    for a in axes[r]:
        a.set_xticks([]); a.set_yticks([])
savefig(fig, "evidence_panels")
model.train(); ema.restore(model)

REPORT = dict(run=RUN_NAME, ablation=ABLATION, verdict=VERDICT, gates=G,
              seed=SEED, manifest_hash=MANIFEST["hash"], epochs_completed=int(last["epoch"]),
              batches=int(last["step"]), minutes=float(last["minutes"]),
              select_metric=SELECT_METRIC, best_select=float(best["select"]),
              best_epoch=int(best["epoch"]),
              best_epoch_metrics={k: (v if not isinstance(v, np.generic) else v.item())
                                  for k, v in best.to_dict().items()},
              efficiency=EFF, config=CFG,
              note="v4 proposed architecture. Validation only. Cross-dataset numbers come from "
                   "the final evaluation cell and were never used for selection.")
Path(OUT_DIR, "report.json").write_text(json.dumps(REPORT, indent=2, default=str))
print(f"\nreport -> {OUT_DIR}/report.json | figures -> {OUT_DIR}/figures/")


GATE CHECK — v4_b_5ep (A4)

                                  GATE CHECK                                  
  Gate                         Status  Detail
  ---------------------------- ------  --------------------------------------


TypeError: cannot unpack non-iterable bool object

In [ ]:
# ============================ FINAL EVALUATION — run once, after freeze ============================
# First and only read of DFDCP and the FF++ manipulated sequences (spec 34, 35).
# Nothing below may be used to change any hyper-parameter or to re-select a checkpoint.
ck_path = Path(OUT_DIR, "checkpoint_best.pt")
assert ck_path.exists(), "train first"
ck = torch.load(ck_path, map_location="cpu", weights_only=False)
model.load_state_dict(ck["model_trainable"], strict=False)     # EMA weights of the selected epoch
model.eval()
print(f"frozen checkpoint: epoch {ck['epoch']} | {SELECT_METRIC}={ck['best_select']:.4f} "
      f"| manifest {ck['manifest_hash']}")
assert ck["manifest_hash"] == MANIFEST["hash"], "checkpoint was trained on a different split"

@torch.no_grad()
def score_videos(records, cond=None, t=T_TEST):
    """Video score = mean over T_TEST/T_FRAMES chunks. `cond` applies a fixed nuisance."""
    ld = DataLoader(TestVideoDataset(records, t), batch_size=1, num_workers=2, pin_memory=True)
    out = []
    for xb, _ in ld:
        if cond is not None and cond != "clean":
            arr = (xb[0].permute(0, 2, 3, 1).numpy() * IMNET_STD + IMNET_MEAN)
            arr = (np.clip(arr, 0, 1) * 255).astype(np.uint8)
            xb = torch.stack([to_tensor(intervene_fixed(a, cond)) for a in arr])[None]
        chunks = []
        for ch in xb.split(T_FRAMES, dim=1):
            with torch.autocast("cuda", dtype=AMP, enabled=device.type == "cuda"):
                o = net(ch.to(device, non_blocking=True),
                        ch.to(device, non_blocking=True),
                        step_fraction=1.0)
            if isinstance(o, tuple):
                o = o[0]
            chunks.append(float(torch.sigmoid(o["logit"].float()).mean()))
        out.append(float(np.mean(chunks)))
    return out

EVAL, TABLES = {}, []

banner("FINAL EVALUATION — 9 test conditions")
print(f"  Frozen checkpoint: epoch {ck['epoch']} | {SELECT_METRIC}={ck['best_select']:.4f}")

# ---- DFDCP cross-dataset ------------------------------------------------
if TEST_SETS.get("dfdcp_real") and TEST_SETS.get("dfdcp_fake"):
    r = score_videos(TEST_SETS["dfdcp_real"])
    f = score_videos(TEST_SETS["dfdcp_fake"])
    y, p = [0] * len(r) + [1] * len(f), r + f
    m = classification_metrics(y, p, thr=0.5)
    m.update(bootstrap_auc(y, p))
    m.update(n_real=len(r), n_fake=len(f))
    EVAL["dfdcp"] = m
    TABLES.append(dict(benchmark="DFDCP (cross-dataset)", n=len(y), auc=m["auc"],
                       ci_lo=m["lo"], ci_hi=m["hi"], ap=m["ap"], eer=m["eer"]))
    print(f"\n  DFDCP  AUROC {m['auc']:.4f} [95% CI {m['lo']:.4f}, {m['hi']:.4f}]  "
          f"AP {m['ap']:.4f}  EER {m['eer']:.4f}  (v1 baseline: 0.5706)")

    # robustness on the unseen domain
    rob = {"clean": m["auc"]}
    for cond in [c for c in ROBUST_CONDS if c != "clean"]:
        rr, ff = score_videos(TEST_SETS["dfdcp_real"], cond), score_videos(TEST_SETS["dfdcp_fake"], cond)
        rob[cond] = float(roc_auc_score([0] * len(rr) + [1] * len(ff), rr + ff))
        print(f"  DFDCP  {cond:<9s}  AUROC {rob[cond]:.4f}")
    EVAL["dfdcp_robustness"] = rob
else:
    print("\n  DFDCP not present - skipped")

# ---- Celeb-DF-v2 cross-dataset ------------------------------------------
if TEST_SETS.get("cdf_test_real") and TEST_SETS.get("cdf_test_fake"):
    r = score_videos(TEST_SETS["cdf_test_real"])
    f = score_videos(TEST_SETS["cdf_test_fake"])
    y, p = [0] * len(r) + [1] * len(f), r + f
    m_cdf = classification_metrics(y, p, thr=0.5)
    m_cdf.update(bootstrap_auc(y, p))
    m_cdf.update(n_real=len(r), n_fake=len(f))
    EVAL["cdf"] = m_cdf
    TABLES.append(dict(benchmark="Celeb-DF-v2 (cross-dataset)", n=len(y), auc=m_cdf["auc"],
                       ci_lo=m_cdf["lo"], ci_hi=m_cdf["hi"], ap=m_cdf["ap"], eer=m_cdf["eer"]))
    print(f"\n  Celeb-DF  AUROC {m_cdf['auc']:.4f} [95% CI {m_cdf['lo']:.4f}, {m_cdf['hi']:.4f}]  "
          f"AP {m_cdf['ap']:.4f}  EER {m_cdf['eer']:.4f}")
else:
    print("\n  Celeb-DF test not present - skipped")

# ---- FF++ cross-manipulation --------------------------------------------
if TEST_SETS.get("ffpp_test_real"):
    base = score_videos(TEST_SETS["ffpp_test_real"])
    cross = {}
    for key, recs in sorted(TEST_SETS.items()):
        if not key.startswith("ffpp_") or key == "ffpp_test_real":
            continue
        manip = key[len("ffpp_"):]
        fs = score_videos(recs)
        y, p = [0] * len(base) + [1] * len(fs), base + fs
        mm = classification_metrics(y, p, thr=0.5)
        mm.update(bootstrap_auc(y, p)); mm["n_fake"] = len(fs)
        cross[manip] = mm
        TABLES.append(dict(benchmark=f"FF++ {manip} (cross-manip)", n=len(y), auc=mm["auc"],
                           ci_lo=mm["lo"], ci_hi=mm["hi"], ap=mm["ap"], eer=mm["eer"]))
        print(f"  FF++ {manip:<18s}  AUROC {mm['auc']:.4f} [{mm['lo']:.4f}, {mm['hi']:.4f}] ({len(fs)} fakes)")
    if cross:
        macro = float(np.mean([v["auc"] for v in cross.values()]))
        EVAL["ffpp_cross_manip"] = cross
        EVAL["ffpp_macro_auc"] = macro
        TABLES.append(dict(benchmark="FF++ macro-average", n=len(cross), auc=macro,
                           ci_lo=float("nan"), ci_hi=float("nan"), ap=float("nan"), eer=float("nan")))
        print(f"\n  FF++ macro-average AUROC {macro:.4f} over {len(cross)} families")
else:
    print("\n  FF++ manipulated sequences not present - skipped")

print_eval_table(TABLES)

EVAL["meta"] = dict(run=RUN_NAME, ablation=ABLATION, checkpoint_epoch=int(ck["epoch"]),
                    select_metric=SELECT_METRIC, select_value=float(ck["best_select"]),
                    manifest_hash=MANIFEST["hash"], t_test=T_TEST,
                    trained_on="DD-SBI fakes from FF++/Celeb-DF reals only; no real manipulated video")
Path(OUT_DIR, "evaluation", "final_results.json").write_text(json.dumps(EVAL, indent=2, default=str))
pd.DataFrame(TABLES).to_csv(Path(OUT_DIR, "evaluation", "table1_main.csv"), index=False)
print(f"  results -> {OUT_DIR}/evaluation/final_results.json")


## Reading the result (v4)

**v4 design principles:**
1. Route by forensic representation (prototype router)
2. Assess trust separately (M/S/R are trust, not routing, variables)
3. Preserve common knowledge (shared expert always evaluated)
4. Train the exact evaluation graph (relative curriculum, no absolute thresholds)

**Selection metric** is the worst-case validation AUROC across `clean / jpeg30 / blur15 / resize50`
on the hard SBI validation split.

**The headline number** is DFDCP AUROC from the final cell. The v3.1 baseline scored 0.5904
with a source validation AUROC of 0.9720 — the 0.3816 gap was caused by train/test graph
mismatch. v4 should show a smaller gap by training the exact graph used for evaluation.

**Ablation stages:**

| Stage | Change | Question |
|-------|--------|----------|
| A0 | Corrected curriculum only | Is train/test graph mismatch the main failure? |
| A1 | + Switch balance + z-loss | Does router stabilization prevent collapse? |
| A2 | + shared expert | Does common knowledge stabilize transfer? |
| A3 | + prototype router | Does representation-first routing reduce shortcuts? |
| A4 | + semantic specialist axes | Does meaningful specialization improve OOD? |

## Outputs

`config.json` `manifest.json` `metrics.{json,csv}` `progress.{json,csv}` `report.json`
`checkpoint_{best,last}.pt` `figures/*.{png,pdf}` `evaluation/final_results.json`
